# 知识

## 滑点
**滑点**（Slippage）是指预期或指令的价格与实际成交价格之间出现的差异。通常由以下因素导致：
- 市场波动性高： 当市场剧烈波动时，从下达指令到订单实际在交易所执行的期间内，价格可能已经发生了变动。
- 流动性不足： 市场上没有足够的买家或卖家来匹配订单，即订单无法立即以期望的价格完全成交，从而需要以下一个可用的价格来完成交易。
- 网络延迟： 交易指令从设备传输到交易服务器，再到交易所，这个过程中存在网络延迟。
- 订单类型：
  - **市价单**（Market Order）：以当前市场最佳可得价格立即执行。
    - 由于市场价格在不断变化，容易出现滑点。
  - **止损单**（Stop-Loss Order）：为了限制损失而设置，当股票价格达到止损价时会触发一个市价单。
    - 如果市场快速下跌，止损单可能无法在设定的价格上成交，而是以更低的价格成交，造成更大的损失。
  - **限价单**（Limit Order）：限价单指令只会在设定的价格或更好的价格成交。
    - 限价单通常不会出现不利的滑点，但缺点是如果价格未能达到您的设定，订单可能无法成交。

# 订单处理

## check_group_lens_nb
生成未成交订单结果对象

参数
- `status`：int，订单状态码，通常表示拒绝或忽略
  - OrderStatus.`Rejected`: 订单被拒绝  
  - OrderStatus.`Ignored`: 订单被忽略
- `status_info`：int，具体的状态信息码，说明拒绝原因
  - OrderStatusInfo.`NoCashLong`：做多资金不足
  - OrderStatusInfo.`NoOpenPosition`：无持仓可平
  - OrderStatusInfo.`SizeZero`：订单大小为零
  - OrderStatusInfo.`MaxSizeExceeded`：超过最大订单限制
  - OrderStatusInfo.`MinSizeNotReached`：未达到最小订单限制
  - OrderStatusInfo.`CantCoverFees`：无法承担手续费
  - OrderStatusInfo.`PartialFill`：部分成交被拒绝

返回：订单结果对象 OrderResult。包含以下字段
- size: np.nan (未成交数量)
- price: np.nan (未成交价格) 
- fees: np.nan (未产生手续费)
- side: -1 (无交易方向)
- status: 传入的状态码
- status_info: 传入的状态详情码

### 源码
```python
@njit(cache=True)
def order_not_filled_nb(status: int, status_info: int) -> OrderResult:
    return OrderResult(np.nan, np.nan, np.nan, -1, status, status_info)
```

## buy_nb
执行买入订单或平空头操作。

### 参数

- `exec_state` : ExecuteOrderState，当前执行状态。包含：
  - `cash`: 总现金。通常是指多个资产，包括空头的保证金
  - `position`: 当前头寸（正数=多头，负数=空头）。针对当前资产
  - `debt`: 空头债务（用于计算平均成本）。针对当前资产，如果其空头，指的是 *空头数 * 空头时成交价*
  - `free_cash`: 可用现金。可供当前资产交易的自由现金
- `size` : float，期望买入数量。可以是：
  - 正数: 具体买入数量
  - np.inf: 使用所有可用资金买入
- `price` : float，目标买入价格。
  - 实际成交价会考虑滑点调整
- `direction` : int，交易方向限制：
    - `Direction.Both`: 允许开多头或平空头
    - `Direction.LongOnly`: 只允许开多头
    - `Direction.ShortOnly`: 只允许平空头
- `fees` : float, 比例手续费率（例如 0.001 表示 0.1%）
- `fixed_fees` : float, 固定手续费（绝对金额）
- `slippage` : float, 滑点率。
  - 买入时向上滑点，实际价格 = price * (1 + slippage)
- `min_size` : float, 最小订单数量。小于此值的订单将被拒绝
- `max_size` : float, 最大订单数量。超过此值的订单将被截断或拒绝
- `size_granularity` : float, 数量粒度。
  - 订单数量将向下取整到此粒度的整数倍。例如：粒度为 0.1，则 1.37 会变为 1.3
- `lock_cash` : bool, 是否锁定现金。
  - 如果为 True：
    - 多头时只能使用 free_cash
    - 空头时需考虑平仓所需资金
- `allow_partial` : bool, 是否允许部分成交。
  - 为 False 时，资金不足的订单将被完全拒绝
- `percent` : float, 资金使用比例。限制最多使用多少比例的可用资金
    
返回：tuple[ExecuteOrderState, OrderResult]，新的执行状态和订单结果
- `ExecuteOrderState`: 更新后的投资组合状态
  - `cash`: 扣除交易成本后的现金
  - `position`: 更新后的头寸
  - `debt`: 更新后的空头债务
  - `free_cash`: 更新后的可用现金
- `OrderResult`: 订单执行结果
  - `size`: 实际成交数量
  - `price`: 实际成交价格（含滑点）
  - `fees`: 实际支付的手续费
  - `side`: OrderSide.Buy
  - `status`: OrderStatus.Filled 或相应的拒绝状态
  - `status_info`: 详细状态信息

### 逻辑

注意：
- 空头时，会锁定 *2倍的空头数×当时成交价* 作为保证金，防止无资金平空头

计算**调整价格** `adj_price = price * (1 + slippage)`
- 因为延时，下达订单时的价格与最终成交时的价格存在差异

计算资金限制 `cash_limit`
- 参数 `lock_cash`
  - 为 `False`：可用所有现金 `cash_limit = exec_state.cash`
  - 为 `True`（只考虑使用 `free_cash` 以及空头时被锁定的保证金）
    - 当前多头 `exec_state.position >= 0`
      - 此时 `cash_limit = exec_state.free_cash`
    - 当前空头
      - 计算完全平仓需要多少现金 `cover_req_cash`
        - $空头总数 \cdot 调整价格 \cdot \left( {1 + 比例手续费率} \right) + 固定手续费$
      - 计算完全平仓后的自由现金 `cover_free_cash=exec_state.free_cash + 2 * exec_state.debt, -cover_req_cash`
        - $当前自由现金  + 释放的保证金 - 完全平仓成本$
      - 如果 `cover_free_cash > 0`（有足够现金平掉当前全部空头）
        - 可用所有现金 `cash_limit = exec_state.free_cash + 2 * exec_state.debt`
      - 如果 `cover_free_cash < 0`
        - 计算空头的平均入场价格 `avg_entry_price = exec_state.debt / abs(exec_state.position)`
        - 计算最多能平空头数 `max_short_size`
          - $自由现金 + 2 \cdot 平均入场价格 \cdot x = 调整价格\left( {1 + 比例手续费率} \right)x + 固定手续费$
        - 资金限制 `cash_limit = max_short_size * adj_price * (1 + fees) + fixed_fees`
      - 否则（即 `cover_free_cash == 0`）
        - `cash_limit=exec_state.free_cash + 2 * exec_state.debt`

考虑参数 `percent` 即比例限制
- `cash_limit = min(cash_limit, percent * cash_limit)`

如果是下述情况，生成未成交订单对象
- 允许开多头
  - `cash_limit = 0`，即无可用现金
  - 期望买入数量 `size` 和 `cash_limit` 都是 `inf`
- 只允许平空头
  - `exec_state.position == 0`：当前无头寸，无法进行平仓操作

计算调整后的订单大小 `adj_size`
- 只允许平空头 
  - `adj_size = min(-exec_state.position, size)`，即订单大小不能超过当前空头头寸的绝对值
- 允许开多头
  - `adj_size = size`
- 根据粒度调整 `adj_size = adj_size // size_granularity * size_granularity`
  - 例如：粒度为 0.1，1.37——>13——>1.3

计算完成此订单的所需现金总额 `total_req_cash`
- `total_req_cash = adj_size * adj_price * (1 + fees) + req_fees`

检查资金是否充足
- 如果 `total_req_cash <= cash_limit`
  - 最终成交数量 `final_size = adj_size `
  - 最终实际支付手续费 `fees_paid = adj_size * adj_price * fees + fixed_fees`
  - 最终实际使用现金 `final_req_cash = total_req_cash`
- 否则（需要减少订单数量以适用资金数 `cash_limit`）
  - 计算 `max_req_cash = (cash_limit - fixed_fees) / (1 + fees)`
    - 如果 `max_req_cash < 0` 即固定手续费都无法承担，返回未成交订单对象
  - 计算最大可购买数 `max_acq_size = max_req_cash / adj_price`
  - 根据粒度 `size_granularity` 调整最大可购买数
  - 确定最终成交数量 `final_size`、实际支付手续费 `fees_paid`、实际使用现金 `final_req_cash`
  
检查，如果是下述情况，返回未成交订单对象
- `adj_size < 0`
- `final_size` 小于参数 `min_size`
- 参数 `size < ∞` 并且 `final_size < size` 并且参数 `allow_partial==False` 

更新
- 现金：`new_cash = exec_state.cash - final_req_cash`
- 头寸：`new_position = exec_state.position + final_size`
- 如果原来是空头 `exec_state.position < 0`
  - 计算买入数量 `short_size`
  - 更新债务 `new_debt`$ =exec\_state.debt - short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}}$
  - 更新新自由现金 `new_free_cash`：$原自由现金 + 释放的债务保证金(2倍) - 交易成本$
    - $exec\_state.free\_cash + 2short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}} - final\_req\_cash$
- 如果原来无空头或者为多头
  - 更新债务 `new_debt = exec_state.debt`
  - 更新新自由现金 `new_free_cash = exec_state.free_cash - final_req_cash`

构建
- 订单结果 `OrderResult`
  - `final_size`：实际成交数量
  - `adj_price`：实际成交价格（含滑点）
  - `fees_paid`：实际支付的手续费
  - `OrderSide.Buy`：订单方向：买入
  - `OrderStatus.Filled`：订单状态：已成交
  - `-1`：状态详情，无特殊信息
- 执行订单状态 `ExecuteOrderState`
  - `cash=new_cash`：更新后的现金余额
  - `position=new_position`：更新后的头寸
  - `debt=new_debt`：更新后的债务
  - `free_cash=new_free_cash`：更新后的可用现金

## sell_nb
执行卖出订单或开空头操作。

### 参数
- `exec_state` : ExecuteOrderState，当前执行状态。包含：
  - `cash`: 总现金。通常是指多个资产，包括空头的保证金
  - `position`: 当前头寸（正数=多头，负数=空头）。针对当前资产
  - `debt`: 空头债务（用于计算平均成本）。针对当前资产，如果其空头，指的是 *空头数 * 空头时成交价*
  - `free_cash`: 可用现金。
- `size`：float，期望卖出数量。可以是：
  - 正数: 具体卖出数量
  - np.inf: 卖出所有持仓或开最大空头
- `price`：float，目标卖出价格。实际成交价会考虑滑点调整
- `direction`：int, 可选 (默认: Direction.Both)
  - `Direction.Both`：允许平多头或开空头
  - `Direction.LongOnly`：只允许平多头
  - `Direction.ShortOnly`：只允许开空头
- `fees`：float, 可选 (默认: 0.0)。比例手续费率（例如 0.001 表示 0.1%）
- `fixed_fees`：float, 可选 (默认: 0.0)。固定手续费（绝对金额）
- `slippage`：float, 可选 (默认: 0.0)，滑点率。
  - 卖出时向下滑点，实际价格 = price * (1 - slippage)
- `min_size`：float, 可选 (默认: 0.0)，最小订单数量。小于此值的订单将被拒绝
- `max_size`：float, 可选 (默认: np.inf)，最大订单数量。超过此值的订单将被截断或拒绝
- `size_granularity`：float, 可选 (默认: np.nan)，数量粒度。
  - 订单数量将向下取整到此粒度的整数倍
- `lock_cash`：bool, 可选 (默认: False)，是否锁定现金。
  - 如果为 True，则限制空头开仓的最大数量
- `allow_partial`：bool, 可选 (默认: True)，是否允许部分成交。
  - 为 False 时，保证金不足的订单将被完全拒绝
- `percent`：float, 可选 (默认: np.nan)，头寸使用比例。限制最多卖出多少比例的可卖数量

返回：tuple[ExecuteOrderState, OrderResult]，新的执行状态和订单结果
- `ExecuteOrderState`：更新后的投资组合状态
  - `cash`：增加卖出收入后的现金（平多头时）或不变（开空头时）
  - `position`：更新后的头寸（减少或变负）
  - `debt`：更新后的空头债务（开空头时增加）
  - `free_cash`：更新后的可用现金（考虑保证金锁定）
- `OrderResult`：订单执行结果
  - `size`：实际成交数量
  - `price`：实际成交价格（含滑点）
  - `fees`：实际支付的手续费
  - `side`：OrderSide.Sell
  - `status`：OrderStatus.Filled 或相应的拒绝状态
  - `status_info`：详细状态信息

### 逻辑

注意：
- 空头时，会锁定 *2倍的空头数×当时成交价* 作为保证金，防止无资金平空头

计算**调整价格** `adj_price = price * (1 - slippage)`
- 因为延时，下达订单时的价格与最终成交时的价格存在差异

计算可卖出的最大数量 `size_limit`
- 只允许多头（参数 `direction == Direction.LongOnly`）
  - 卖出数量不能超过当前多头头寸 `size_limit = min(exec_state.position, size)`
- 允许开空头
  - 如果 `lock_cash=True` 或者 `size` 是 `inf` 以及 `percent` 是 `Nan`
    - 计算总可用现金 `total_free_cash`
      - $自由现金 + \max \left( {头寸数,0} \right) \cdot 调整价格\left( {1 - 比例手续费率} \right)$
    - 如果 `total_free_cash <= 0`
      - 如果当前头寸数 `exec_state.position <= 0`：返回未成交订单对象
      - 只能平现有多头：`max_size_limit = max(exec_state.position, 0)  `
    - 否则
      - 计算最大可开空头数量 `max_short_size` $= \frac{{总可用现金 - 固定手续费}}{{调整价格\left( {1 + 比例手续费率} \right)}}$
        - 开空头只需要保证金与手续费，这里这样限制是为了防止过度杠杆即之后没现金平空头
      - `max_size_limit =  max(exec_state.position, 0) + max_short_size`
      - 如果 `max_size_limit <= 0`：返回未成交订单对象
    - 如果 `lock_cash=True`
      - 如果 `size` 是 `inf` 并且 `percent` 不是 `Nan`
        - `size_limit = min(percent * max_size_limit, max_size_limit)`
        - `percent = np.nan`
      - 否则如果 `percent` 不是 `Nan`
        - `size_limit = min(percent * size, max_size_limit)`
        - `percent = np.nan`
      - 否则（`percent` 是 `Nan`）
        - `size_limit = max_size_limit`

  - 否则
    - 使用原始订单大小 `size_limit = size`

考虑参数 `percent` 即比例限制
- 如果 `precent` 不是 `Nan`：`size_limit = percent * size_limit`

考虑最大订单限制
- 如果 `size_limit > max_size` 
  - 如果允许部分成交 `allow_partial`：`size_limit = max_size`
  - 否则，生成未成交订单对象

如果允许开空头且 `size_limit` 为 `inf`：即无限大空头开仓，报错

如果只允许平多头且当前无头寸：生成未成交订单对象

根据粒度调整 `size_limit = size_limit // size_granularity * size_granularity`
- 例如：粒度为 0.1，1.37——>13——>1.3

检查，如果是下述情况，生成未成交订单对象
- `size_limit < 0`
- `size_limit < min_size`
- `size` 有限且 `size_limit < size` 且不允许部分成交

计算卖出获得的总现金 `acq_cash`、手续费 `fees_paid`、扣除手续费后的现金收入
- `acq_cash = size_limit * adj_price`
- `fees_paid = acq_cash * fees + fixed_fees`
- `final_acq_cash = acq_cash - fees_paid`

更新
- 现金：`new_cash = exec_state.cash + final_acq_cash`
- 头寸：`new_position = exec_state.position - size_limit`
- 如果变成空头 `new_position < 0`
  - 计算空头数量 `short_size`
    - 原来是空头：`short_size = size_limit`（原来就是空头，增加空头规模）
    - 否则：`short_size = abs(new_position)`
  - 更新债务 `new_debt`$ =exec\_state.debt + short\_value$
  - 更新新自由现金 `new_free_cash`：$原自由现金 - 2倍空头价值 + final\_acq\_cash$
- 否则
  - 更新债务 `new_debt = exec_state.debt`
  - 更新新自由现金 `new_free_cash = exec_state.free_cash + final_acq_cash`

构建
- 订单结果 `OrderResult`
  - `final_size`：实际成交数量
  - `adj_price`：实际成交价格（含滑点）
  - `fees_paid`：实际支付的手续费
  - `OrderSide.Buy`：订单方向：卖出
  - `OrderStatus.Filled`：订单状态：已成交
  - `-1`：状态详情，无特殊信息
- 执行订单状态 `ExecuteOrderState`
  - `cash=new_cash`：更新后的现金余额
  - `position=new_position`：更新后的头寸
  - `debt=new_debt`：更新后的债务
  - `free_cash=new_free_cash`：更新后的可用现金

## execute_order_nb
根据订单 `order: Order` 和投资组合状态 `state: ProcessOrderState`执行 `buy_nb` 或 `sell_nb`。

### 参数和返回
参数
- `state`：ProcessOrderState。当前的投资组合处理状态
  - `cash`：float，现金余额：当前列或现金共享组的现金总额
  - `position`：float，持仓数量：当前列的资产持仓数量（正数多头，负数空头）
  - `debt`：float，做空债务：当前列做空操作产生的债务总额
  - `free_cash`：float，可用现金：当前列或现金共享组的可用于交易的现金
  - `val_price`：float，估值价格：当前列资产的估值价格（用于价值计算）
  - `value`：float，总价值：当前列或现金共享组的总价值（现金+持仓价值-债务）
  - `oidx`：int，单索引：对应的订单记录在order_records数组中的索引位置
  - `lidx`：int，日志索引：对应的日志记录在log_records数组中的索引位置   
- `order`：Order，待执行的订单对象
  - `size`：float = np.inf，订单大小：要交易的数量或金额
  - `price`：float = np.inf，订单价格：每单位的交易价格
  - `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
  - `direction`：int = Direction.Both，允许方向：订单允许的交易方向
  - `fees`：float = 0.0，手续费率：按订单价值的百分比收费
  - `fixed_fee`：float = 0.0，固定手续费：每笔订单的固定费用
  - `slippage`：float = 0.0，滑点率：价格滑动的百分比
  - `min_size`：float = 0.0，最小大小：订单的最小允许大小
  - `max_size`：float = np.inf，最大大小：订单的最大允许大小
  - `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
  - `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
  - `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
  - `allow_partial`：bool = True，允许部分成交：是否接受部分填充
  - `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
  - `log`：bool = False，日志记录：是否记录此订单的详细日志
  
返回：tp.Tuple[ExecuteOrderState, OrderResult]，订单执行状态和订单结果的元组
- 执行订单状态 `ExecuteOrderState`
  - `cash`：float，现金余额：订单执行后的现金总额
  - `position`：float，持仓数量：订单执行后的持仓数量
  - `debt`：float，做空债务：订单执行后的债务总额
  - `free_cash`：float，可用现金：订单执行后的可用现金
- 订单结果 `OrderResult`
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）

### 逻辑
（1）获取投资组合状态 `state` 中的各成分，接近零时设为精确零
- `cash`、`position`、`debt`、`free_cash`、`val_price`、`value`

（2）预构建订单执行状态 `exec_state`
- `cash`、`position`、`debt`、`free_cash`

（3）检查订单 `order` 和投资组合状态 `state` 中各成分的合法性

（4）计算订单大小 `order_size`
- `order_size = order.size`，订单大小类型 `order_size_type = order.size_type`
- 如果是只做空订单（`order.direction == Direction.ShortOnly`）
  - `order_size *= -1`
  - 这是为了表达如下的语义：正数表示增加头寸，负数表示减少头寸
- 如果 `order_size_type` 为 `SizeType.TargetPercent`（目标百分比模式）
  - 转换为目标价值
  - `order_size *= value`
  - `order_size_type = SizeType.TargetValue`
- 如果 `order_size_type` 为 `SizeType.Value` 或者 `SizeType.TargetValue`
  - 将价值转换为数量：`order_size /= val_price`
  - 如果 `order_size_type` 为 `SizeType.Value`，则变为 `SizeType.Amount`，否则 `SizeType.TargetAmount`
- 如果 `order_size_type` 为 `SizeType.TargetAmount`
  - 计算需要交易的数量 = 目标数量 - 当前持有数量
  - `order_size -= position`
  - `order_size_type = SizeType.Amount` 
- 如果 `order_size_type` 为 `SizeType.Percent`（基于当前可用资源的百分比）
  - `percent = abs(order_size)`，获取去符号的百分比值
  - `order_size = np.sign(order_size) * np.inf`，设置为带符号的无限大
  - `order_size_type = SizeType.Amount`

（5）根据 `order_size` 的符号执行买入或卖出操作，并返回相应结果
- `order_size > 0`：执行 `buy_nb`
- `order_size <= 0`：执行 `sell_nb`
- 参数
  - `exec_state`：当前执行状态
  - `+/-order_size`：买入/卖出数量
  - `order.price`：买入/卖出价格
  - `direction=order.direction`：交易方向限制
  - `fees=order.fees`：手续费率
  - `fixed_fees=order.fixed_fees`：固定手续费
  - `slippage=order.slippage`：滑点设置
  - `min_size=order.min_size`：最小订单大小
  - `max_size=order.max_size`：最大订单大小
  - `size_granularity=order.size_granularity`：数量粒度
  - `lock_cash=order.lock_cash`：是否锁定现金
  - `allow_partial=order.allow_partial`：是否允许部分成交
  - `percent=percent`：资金使用百分比

## fill_log_record_nb

将订单执行过程中的所有关键信息记录到日志记录中 `record`。

参数：
- `record`：Record，待填充的日志记录对象
- `record_id`：int，记录唯一标识符
- `i`：int，时间索引（通常是时间步或K线索引）
- `col`：int，列索引（资产标识符）
- `group`：int，组标识符（用于资产分组）
- `cash`：float，执行前的现金余额
- `position`：float，执行前的头寸
- `debt`：float，执行前的债务
- `free_cash`：float，执行前的可用现金
- `val_price`：float，执行前的估值价格
- `value`：float，执行前的组合价值
- `order`：Order，原始订单对象
- `new_cash`：float，执行后的现金余额
- `new_position`：float，执行后的头寸
- `new_debt`：float，执行后的债务
- `new_free_cash`：float，执行后的可用现金
- `new_val_price`：float，执行后的估值价格
- `new_value`：float，执行后的组合价值
- `order_result`：OrderResult，订单执行结果
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）
- `order_id`：int，订单标识符

### 源码
```python
@njit(cache=True)
def fill_log_record_nb(record: tp.Record,
                       record_id: int,
                       i: int,
                       col: int,
                       group: int,
                       cash: float,
                       position: float,
                       debt: float,
                       free_cash: float,
                       val_price: float,
                       value: float,
                       order: Order,
                       new_cash: float,
                       new_position: float,
                       new_debt: float,
                       new_free_cash: float,
                       new_val_price: float,
                       new_value: float,
                       order_result: OrderResult,
                       order_id: int) -> None:

    record['id'] = record_id
    record['group'] = group
    record['col'] = col
    record['idx'] = i
    record['cash'] = cash
    record['position'] = position
    record['debt'] = debt
    record['free_cash'] = free_cash
    record['val_price'] = val_price
    record['value'] = value
    record['req_size'] = order.size
    record['req_price'] = order.price
    record['req_size_type'] = order.size_type
    record['req_direction'] = order.direction
    record['req_fees'] = order.fees
    record['req_fixed_fees'] = order.fixed_fees
    record['req_slippage'] = order.slippage
    record['req_min_size'] = order.min_size
    record['req_max_size'] = order.max_size
    record['req_size_granularity'] = order.size_granularity
    record['req_reject_prob'] = order.reject_prob
    record['req_lock_cash'] = order.lock_cash
    record['req_allow_partial'] = order.allow_partial
    record['req_raise_reject'] = order.raise_reject
    record['req_log'] = order.log
    record['new_cash'] = new_cash
    record['new_position'] = new_position
    record['new_debt'] = new_debt
    record['new_free_cash'] = new_free_cash
    record['new_val_price'] = new_val_price
    record['new_value'] = new_value
    record['res_size'] = order_result.size
    record['res_price'] = order_result.price
    record['res_fees'] = order_result.fees
    record['res_side'] = order_result.side
    record['res_status'] = order_result.status
    record['res_status_info'] = order_result.status_info
    record['order_id'] = order_id
```

## fill_order_record_nb
将成功执行的订单结果填充到订单记录 `record` 中。
- 与 `fill_log_record_nb` 不同，该函数只记录核心的订单执行结果，不包括详细的状态变化和请求参数。

参数
- `record`：Record，待填充的订单记录对象
- `record_id`：int，记录唯一标识符
- `i`：int，
- `col`：int，列索引（资产标识符）
- `order_result`：OrderResult，订单执行结果对象
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）

### 源码
```python
@njit(cache=True)
def fill_order_record_nb(record: tp.Record,
                         record_id: int,
                         i: int,
                         col: int,
                         order_result: OrderResult) -> None:

    record['id'] = record_id
    record['col'] = col
    record['idx'] = i
    record['size'] = order_result.size
    record['price'] = order_result.price
    record['fees'] = order_result.fees
    record['side'] = order_result.side
```

## raise_rejected_order_nb
根据订单结果 `order_result: OrderResult` 抛出订单拒绝异常 `RejectedOrderError`。

### 源码
```python
@njit(cache=True)
def raise_rejected_order_nb(order_result: OrderResult) -> None:

    if order_result.status_info == OrderStatusInfo.SizeNaN:
        raise RejectedOrderError("Size is NaN")
    if order_result.status_info == OrderStatusInfo.PriceNaN:
        raise RejectedOrderError("Price is NaN")
    if order_result.status_info == OrderStatusInfo.ValPriceNaN:
        raise RejectedOrderError("Asset valuation price is NaN")
    if order_result.status_info == OrderStatusInfo.ValueNaN:
        raise RejectedOrderError("Asset/group value is NaN")
    if order_result.status_info == OrderStatusInfo.ValueZeroNeg:
        raise RejectedOrderError("Asset/group value is zero or negative")
    if order_result.status_info == OrderStatusInfo.SizeZero:
        raise RejectedOrderError("Size is zero")
    if order_result.status_info == OrderStatusInfo.NoCashShort:
        raise RejectedOrderError("Not enough cash to short")
    if order_result.status_info == OrderStatusInfo.NoCashLong:
        raise RejectedOrderError("Not enough cash to long")
    if order_result.status_info == OrderStatusInfo.NoOpenPosition:
        raise RejectedOrderError("No open position to reduce/close")
    if order_result.status_info == OrderStatusInfo.MaxSizeExceeded:
        raise RejectedOrderError("Size is greater than maximum allowed")
    if order_result.status_info == OrderStatusInfo.RandomEvent:
        raise RejectedOrderError("Random event happened")
    if order_result.status_info == OrderStatusInfo.CantCoverFees:
        raise RejectedOrderError("Not enough cash to cover fees")
    if order_result.status_info == OrderStatusInfo.MinSizeNotReached:
        raise RejectedOrderError("Final size is less than minimum allowed")
    if order_result.status_info == OrderStatusInfo.PartialFill:
        raise RejectedOrderError("Final size is less than requested")
    raise RejectedOrderError
```

## update_value_nb
更新估值价格和投资组合总价值
- 每次订单执行后，需要更新资产的估值价格和投资组合的总价值

参数
- `cash_before`：float，订单执行前的现金余额
- `cash_now`：float，订单执行后的现金余额  
- `position_before`：float，订单执行前的头寸数量
- `position_now`：float，订单执行后的头寸数量
- `val_price_before`：float，订单执行前的资产估值价格
- `price`：float，订单执行时的实际成交价格（用作新的估值价格）
- `value_before`：float，订单执行前的投资组合总价值

逻辑
- 新估值价格 = 订单最新成交价格 `price`
- 现金流变化 = 执行后现金 - 执行前现金
- 资产价值变化 = 新头寸 × 新价格 - 旧头寸 × 旧价格
- 新总价值 = 旧总价值 + 资产价值变化 + 现金流变化
    
返回：`tuple[float, float]` (新估值价格, 新总价值)

### 源码
```python
@njit(cache=True)
def update_value_nb(cash_before: float,
                    cash_now: float,
                    position_before: float,
                    position_now: float,
                    val_price_before: float,
                    price: float,
                    value_before: float) -> tp.Tuple[float, float]:
    # 更新估值价格为最新成交价格
    val_price_now = price
    
    # 计算现金流变化（现金的增减）
    cash_flow = cash_now - cash_before
    
    # 计算订单执行前的资产价值
    if position_before != 0:
        asset_value_before = position_before * val_price_before  # 旧头寸 * 旧价格
    else:
        asset_value_before = 0.  # 无头寸时资产价值为零
    
    # 计算订单执行后的资产价值
    if position_now != 0:
        asset_value_now = position_now * val_price_now  # 新头寸 * 新价格
    else:
        asset_value_now = 0.  # 无头寸时资产价值为零
    
    # 计算资产价值的变化
    asset_value_diff = asset_value_now - asset_value_before
    
    # 计算新的投资组合总价值
    # 新总价值 = 原总价值 + 现金变化 + 资产价值变化
    value_now = value_before + cash_flow + asset_value_diff
    
    return val_price_now, value_now
```

## process_order_nb
订单处理的最高层包装函数，整合了订单执行、记录保存和状态更新的完整流程

### 参数和返回
- `i`：int，时间索引（当前时间步或K线索引）
- `col`：int，资产列索引（资产标识符）
- `group`：int，资产组索引（用于分组管理）
- `state`：ProcessOrderState，当前的处理状态（包含现金、头寸、债务等信息）
  - `cash`：float，现金余额：当前列或现金共享组的现金总额
  - `position`：float，持仓数量：当前列的资产持仓数量（正数多头，负数空头）
  - `debt`：float，做空债务：当前列做空操作产生的债务总额
  - `free_cash`：float，可用现金：当前列或现金共享组的可用于交易的现金
  - `val_price`：float，估值价格：当前列资产的估值价格（用于价值计算）
  - `value`：float，总价值：当前列或现金共享组的总价值（现金+持仓价值-债务）
  - `oidx`：int，订单索引：对应的订单记录在 `order_records` 数组中的索引位置
  - `lidx`：int，日志索引：对应的日志记录在 `log_records` 数组中的索引位置
- `update_value`：bool，是否在订单成交后更新投资组合价值
- `order`：Order，待处理的订单对象
  - `size`：float = np.inf，订单大小：要交易的数量或金额
  - `price`：float = np.inf，订单价格：每单位的交易价格
  - `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
  - `direction`：int = Direction.Both，允许方向：订单允许的交易方向
  - `fees`：float = 0.0，手续费率：按订单价值的百分比收费
  - `fixed_fees`：float = 0.0，固定手续费：每笔订单的固定费用
  - `slippage`：float = 0.0，滑点率：价格滑动的百分比
  - `min_size`：float = 0.0，最小大小：订单的最小允许大小
  - `max_size`：float = np.inf，最大大小：订单的最大允许大小
  - `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
  - `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
  - `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
  - `allow_partial`：bool = True，允许部分成交：是否接受部分填充
  - `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
  - `log`：bool = False，日志记录：是否记录此订单的详细日志
- `order_records`：RecordArray，用于存储成功订单记录的数组
- `log_records`：RecordArray，用于存储详细日志记录的数组
    
返回：`tuple[OrderResult, ProcessOrderState]` (订单执行结果, 更新后的处理状态)

### 流程
- 使用参数 `state: ProcessOrderState` 和 `order: Order` 调用 `execute_order_nb` 执行订单
  - 返回 `tp.Tuple[ExecuteOrderState, OrderResult]`
- 使用 `raise_rejected_order_nb` 检查订单执行结果，决定是否需要抛出订单拒绝异常 `RejectedOrderError`
- 如果订单成交且需要更新价值，使用 `update_value_nb` 更新估值价格和投资组合总价值
- 如果订单成交，则使用 `fill_order_record_nb` 将结果保存到 `order_records`
- 如果需要记录日志，则使用 `fill_log_record_nb` 将详细信息保存到 `log_records`
- 构造并返回更新后的处理状态 `tuple[OrderResult, ProcessOrderState]`

### 源码
```python
@njit(cache=True)
def process_order_nb(i: int,
                     col: int,
                     group: int,
                     state: ProcessOrderState,
                     update_value: bool,
                     order: Order,
                     order_records: tp.RecordArray,
                     log_records: tp.RecordArray) -> tp.Tuple[OrderResult, ProcessOrderState]:

    exec_state, order_result = execute_order_nb(state, order)

    is_rejected = order_result.status == OrderStatus.Rejected
    if is_rejected and order.raise_reject:
        raise_rejected_order_nb(order_result)

    is_filled = order_result.status == OrderStatus.Filled
    if is_filled and update_value:
        new_val_price, new_value = update_value_nb(
            state.cash,
            exec_state.cash,
            state.position,
            exec_state.position,
            state.val_price,
            order_result.price,
            state.value
        )
    else:
        new_val_price = state.val_price
        new_value = state.value

    new_oidx = state.oidx
    if is_filled:
        # Fill order record
        if state.oidx > len(order_records) - 1:
            raise IndexError("order_records index out of range. Set a higher max_orders.")
        fill_order_record_nb(
            order_records[state.oidx],
            state.oidx,
            i,
            col,
            order_result
        )
        new_oidx += 1

    new_lidx = state.lidx
    if order.log:
        # Fill log record
        if state.lidx > len(log_records) - 1:
            raise IndexError("log_records index out of range. Set a higher max_logs.")
        fill_log_record_nb(
            log_records[state.lidx],
            state.lidx,
            i,
            col,
            group,
            state.cash,
            state.position,
            state.debt,
            state.free_cash,
            state.val_price,
            state.value,
            order,
            exec_state.cash,
            exec_state.position,
            exec_state.debt,
            exec_state.free_cash,
            new_val_price,
            new_value,
            order_result,
            state.oidx if is_filled else -1
        )
        new_lidx += 1

    new_state = ProcessOrderState(
        cash=exec_state.cash,
        position=exec_state.position,
        debt=exec_state.debt,
        free_cash=exec_state.free_cash,
        val_price=new_val_price,
        value=new_value,
        oidx=new_oidx,
        lidx=new_lidx
    )

    return order_result, new_state
```

## order_nb
创建订单对象。

参数
- `size`：float = np.inf，订单大小：要交易的数量或金额
- `price`：float = np.inf，订单价格：每单位的交易价格
- `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
- `direction`：int = Direction.Both，允许方向：订单允许的交易方向
- `fees`：float = 0.0，手续费率：按订单价值的百分比收费
- `fixed_fees`：float = 0.0，固定手续费：每笔订单的固定费用
- `slippage`：float = 0.0，滑点率：价格滑动的百分比
- `min_size`：float = 0.0，最小大小：订单的最小允许大小
- `max_size`：float = np.inf，最大大小：订单的最大允许大小
- `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
- `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
- `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
- `allow_partial`：bool = True，允许部分成交：是否接受部分填充
- `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
- `log`：bool = False，日志记录：是否记录此订单的详细日志

```python
@njit(cache=True)
def order_nb(size: float = np.nan,
             price: float = np.inf,
             size_type: int = SizeType.Amount,
             direction: int = Direction.Both,
             fees: float = 0.,
             fixed_fees: float = 0.,
             slippage: float = 0.,
             min_size: float = 0.,
             max_size: float = np.inf,
             size_granularity: float = np.nan,
             reject_prob: float = 0.,
             lock_cash: bool = False,
             allow_partial: bool = True,
             raise_reject: bool = False,
             log: bool = False) -> Order:

    return Order(
        size=float(size),
        price=float(price),
        size_type=int(size_type),
        direction=int(direction),
        fees=float(fees),
        fixed_fees=float(fixed_fees),
        slippage=float(slippage),
        min_size=float(min_size),
        max_size=float(max_size),
        size_granularity=float(size_granularity),
        reject_prob=float(reject_prob),
        lock_cash=bool(lock_cash),
        allow_partial=bool(allow_partial),
        raise_reject=bool(raise_reject),
        log=bool(log)
    )

```

## close_position_nb
使用 `order_nb` 创建平仓订单。
- 无论当前是多头还是空头头寸，都会创建完全平掉当前持仓的订单。

参数
- `price`：float，可选 (默认: np.inf)，平仓价格，np.inf 表示市价平仓
- `fees`：float, 可选 (默认: 0.0)，比例手续费率
- `fixed_fees`：float, 可选 (默认: 0.0)，固定手续费
- `slippage`：float, 可选 (默认: 0.0)，滑点率
- `min_size`：float, 可选 (默认: 0.0)，最小订单大小
- `max_size`：float, 可选 (默认: np.inf)，最大订单大小
- `size_granularity`：float, 可选 (默认: np.nan)，订单数量粒度
- `reject_prob`：float, 可选 (默认: 0.0)，随机拒绝概率
- `lock_cash`：bool, 可选 (默认: False)，是否锁定现金
- `allow_partial`：bool, 可选 (默认: True)，是否允许部分成交
- `raise_reject`：bool, 可选 (默认: False)，订单被拒绝时是否抛出异常
- `log`：bool, 可选 (默认: False)，是否记录详细日志
    
返回：Order，目标持仓为 0 的订单对象

```python
@njit(cache=True)
def close_position_nb(price: float = np.inf,
                      fees: float = 0.,
                      fixed_fees: float = 0.,
                      slippage: float = 0.,
                      min_size: float = 0.,
                      max_size: float = np.inf,
                      size_granularity: float = np.nan,
                      reject_prob: float = 0.,
                      lock_cash: bool = False,
                      allow_partial: bool = True,
                      raise_reject: bool = False,
                      log: bool = False) -> Order:

    return order_nb(
        size=0.,
        price=price,
        size_type=SizeType.TargetAmount,
        direction=Direction.Both,
        fees=fees,
        fixed_fees=fixed_fees,
        slippage=slippage,
        min_size=min_size,
        max_size=max_size,
        size_granularity=size_granularity,
        reject_prob=reject_prob,
        lock_cash=lock_cash,
        allow_partial=allow_partial,
        raise_reject=raise_reject,
        log=log
    )
```

## order_nothing_nb
创建空订单：返回一个预定义的空订单对象。

```python
@njit(cache=True)
def order_nothing_nb() -> Order:
    return NoOrder
```

# 参数检查

## check_group_lens_nb
检查资产分组长度数组的有效性：验证 `group_lens` 数组的总和是否等于总列数 `n_cols`，确保资产分组配置正确。

参数
- `group_lens`：Array1d
  - 各组的列数数组，每个元素表示对应组包含的资产数量
  - 例如：[2, 3, 1] 表示第1组有2个资产，第2组有3个资产，第3组有1个资产
- `n_cols`：int
    总列数（总资产数量）

```python
@njit(cache=True)
def check_group_lens_nb(group_lens: tp.Array1d, n_cols: int) -> None:
    if np.sum(group_lens) != n_cols:
        raise ValueError("group_lens has incorrect total number of columns")
```

## check_group_init_cash_nb
检查初始现金配置的有效性

验证初始现金数组的长度是否与现金共享模式和资产分组配置匹配。
这确保了每个资产或资产组都有正确的初始现金分配。

参数
- `group_lens`：Array1d，各组的列数数组
- `n_cols`：int，总列数（总资产数量）
- `init_cash`：Array1d，初始现金数组
- `cash_sharing`：bool，现金共享模式标志
  - `True`：组内资产共享现金，`init_cash` 长度应等于组数
  - `False`：每个资产独立现金，`init_cash` 长度应等于资产数

```python
@njit(cache=True)
def check_group_init_cash_nb(group_lens: tp.Array1d, n_cols: int, init_cash: tp.Array1d, cash_sharing: bool) -> None:
    if cash_sharing:
        if len(init_cash) != len(group_lens):
            raise ValueError("If cash sharing is enabled, init_cash must match the number of groups")
    else:
        if len(init_cash) != n_cols:
            raise ValueError("If cash sharing is disabled, init_cash must match the number of columns")
```

## is_grouped_nb
判断是否存在包含多个资产的组。
    
参数
- `group_lens`：Array1d，各组的列数数组，每个元素表示对应组的资产数量
        

```python
@njit(cache=True)
def is_grouped_nb(group_lens: tp.Array1d) -> bool:
    return np.any(group_lens > 1)
```

# 调用序列管理

## shuffle_call_seq_nb
根据 `group_lens` 随机打乱调用序列数组 `call_seq`。
    
参数
- `call_seq`：Array2d，调用序列数组，形状为(时间步, 资产)
  - 每行表示在该时间步的资产调用顺序
- `group_lens`：Array1d 各组的长度数组
  - 每个元素表示对应组包含的资产数量
    
就地修改：直接修改传入的 `call_seq` 数组，不返回新数组

```python
@njit(cache=True)
def shuffle_call_seq_nb(call_seq: tp.Array2d, group_lens: tp.Array1d) -> None:
    from_col = 0
    # 遍历每个资产组
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        # 对每个时间步在当前组内进行随机打乱
        for i in range(call_seq.shape[0]):
            np.random.shuffle(call_seq[i, from_col:to_col]) # 组内随机打乱
        # 移动到下一组
        from_col = to_col
```

In [ ]:
import numpy as np
np.random.seed(42)  # 设置随机种子便于重现
from vectorbt.portfolio.nb import shuffle_call_seq_nb
 
# 创建初始调用序列：3个时间步，6个资产，分为2组[2,4]
call_seq = np.array([[0, 1, 2, 3, 4, 5],
                     [0, 1, 2, 3, 4, 5],
                     [0, 1, 2, 3, 4, 5]])
group_lens = np.array([2, 4])  # 第1组2个资产(0,1)，第2组4个资产(2,3,4,5)
 
print("打乱前:")
print(call_seq)
shuffle_call_seq_nb(call_seq, group_lens)
print("打乱后:")
print(call_seq)  # 组内顺序被随机打乱，但组间边界保持

## build_call_seq_nb
构建新的调用序列数组。

参数
- `target_shape`：Shape，目标形状 (时间步数, 资产数)
- `group_lens`：Array1d，各组的长度数组
- `call_seq_type`：int，可选 (默认: CallSeqType.Default)，调用序列类型
  - `CallSeqType.Default`：正常顺序 (0, 1, 2, ...)
  - `CallSeqType.Reversed`：反向顺序 (最后一个资产优先)
  - `CallSeqType.Random`：随机顺序
    
返回：Array2d，调用序列数组，形状为 `target_shape`

```python
@njit(cache=True)
def build_call_seq_nb(target_shape: tp.Shape,
                      group_lens: tp.Array1d,
                      call_seq_type: int = CallSeqType.Default) -> tp.Array2d:
    if call_seq_type == CallSeqType.Reversed:
        out = np.full(target_shape[1], 1, dtype=np.int64)
        out[np.cumsum(group_lens)[1:] - group_lens[1:] - 1] -= group_lens[1:]
        out = np.cumsum(out[::-1])[::-1] - 1
        out = out * np.ones((target_shape[0], 1), dtype=np.int64)
        return out
    out = np.full(target_shape[1], 1, dtype=np.int64)
    out[np.cumsum(group_lens)[:-1]] -= group_lens[:-1]
    out = np.cumsum(out) - 1
    out = out * np.ones((target_shape[0], 1), dtype=np.int64)
    if call_seq_type == CallSeqType.Random:
        shuffle_call_seq_nb(out, group_lens)
    return out
```

In [ ]:
from vectorbt.portfolio.nb import build_call_seq_nb, CallSeqType

# 创建3个时间步，6个资产，分2组[3,3]的调用序列
target_shape = (3, 6)
group_lens = np.array([3, 3])
 
# 默认顺序
default_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Default)
print("默认顺序:")
print(default_seq)
 
# 反向顺序
reversed_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Reversed)
print("反向顺序:")
print(reversed_seq)

# 随机顺序
np.random.seed(42)
random_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Random)
print("随机顺序:")
print(random_seq)  # 组内随机排列

## require_call_seq
确保调用序列数组 `call_seq` 具有正确的数据类型和内存布局，来满足后续 Numba 编译函数的严格要求

参数
- `call_seq`：Array2d，调用序列数组
    
返回：`Array2d`，满足要求的调用序列数组
    
内存要求
- `dtype=np.int64`：64 位整数类型
- requirements
  - `'A'`：对齐 (Aligned)
  - `'O'`：拥有数据 (Owndata)  
  - `'W'`：可写 (Writeable)
  - `'F'`：Fortran 风格连续

```python
def require_call_seq(call_seq: tp.Array2d) -> tp.Array2d:
    return np.require(call_seq, dtype=np.int64, requirements=['A', 'O', 'W', 'F'])
```

## build_call_seq
构建调用序列数组（非编译优化版本），`build_call_seq_nb` 的非编译版本。
- 使用 NumPy 的向量化操作实现更快的执行速度，以满足后续 Numba 编译函数的要求。

参数
- `target_shape`：Shape，目标形状 (时间步数, 资产数)
- `group_lens`：Array1d，各组的长度数组
- `call_seq_type`：int, 可选 (默认: CallSeqType.Default)，调用序列类型
  - `CallSeqType.Default`：正常顺序 (0, 1, 2, ...)
  - `CallSeqType.Reversed`：反向顺序 (最后一个资产优先)
  - `CallSeqType.Random`：随机顺序
    
返回：Array2d，调用序列数组，满足内存和类型要求

```python
def build_call_seq(target_shape: tp.Shape,
                   group_lens: tp.Array1d,
                   call_seq_type: int = CallSeqType.Default) -> tp.Array2d:
    call_seq = np.full(target_shape[1], 1, dtype=np.int64)
    if call_seq_type == CallSeqType.Reversed:
        call_seq[np.cumsum(group_lens)[1:] - group_lens[1:] - 1] -= group_lens[1:]
        call_seq = np.cumsum(call_seq[::-1])[::-1] - 1
    else:
        call_seq[np.cumsum(group_lens[:-1])] -= group_lens[:-1]
        call_seq = np.cumsum(call_seq) - 1
    call_seq = np.broadcast_to(call_seq, target_shape)
    if call_seq_type == CallSeqType.Random:
        call_seq = require_call_seq(call_seq)
        shuffle_call_seq_nb(call_seq, group_lens)
    return require_call_seq(call_seq)
```

# 辅助工具函数

## get_col_elem_nb
根据上下文对象 `ctx` 和列索引 `col` 从数组 `a` 获取对应元素。

参数
- `ctx`：`Union[OrderContext, PostOrderContext, SignalContext]`，上下文对象，包含：
  - `i`：当前时间步索引
  - `flex_2d`：是否启用2D灵活索引
- `col`：int，目标列索引（资产索引）
- `a`：ArrayLike，待查询的数组
    
返回：Scalar，指定位置的元素值

```python
@njit(cache=True)
def get_col_elem_nb(ctx: tp.Union[RowContext, SegmentContext, FlexOrderContext], col: int,
                    a: tp.ArrayLike) -> tp.Scalar:
    return flex_select_auto_nb(a, ctx.i, col, ctx.flex_2d)
```

## get_elem_nb
根据上下文对象 `ctx` 从数组 `a` 获取对应元素。

参数
- `ctx`：`Union[OrderContext, PostOrderContext, SignalContext]`，上下文对象，包含：
  - `i`：当前时间步索引
  - `col`：当前列索引（资产索引）
  - `flex_2d`：是否启用2D灵活索引
- `a`：ArrayLike，待查询的数组
    
返回：Scalar，指定位置的元素值

```python
@njit(cache=True)
def get_elem_nb(ctx: tp.Union[OrderContext, PostOrderContext, SignalContext],
                a: tp.ArrayLike) -> tp.Scalar:
    return flex_select_auto_nb(a, ctx.i, ctx.col, ctx.flex_2d)
```

## get_group_value_nb
计算指定资产组的当前总价值，包括现金和所有持仓资产的市值。
- $cash\_now{\rm{ }} + \sum\limits_{i \in \left[ {from\_col,to\_col} \right)} {last\_position\left[ i \right]last\_val\_price\left[ i \right]}$

参数
- `from_col`：int，资产组起始列索引（包含）
- `to_col`：int，资产组结束列索引（不包含）
- `cash_now`：float，当前现金余额
- `last_position`：Array1d，最新的持仓数量数组，每个元素对应一个资产
- `last_val_price`：Array1d，最新的估值价格数组，每个元素对应一个资产的当前价格
    
返回：float，资产组的总价值 = 现金 + 所有持仓的市值总和

```python
@njit(cache=True)
def get_group_value_nb(from_col: int,
                       to_col: int,
                       cash_now: float,
                       last_position: tp.Array1d,
                       last_val_price: tp.Array1d) -> float:
    group_value = cash_now
    group_len = to_col - from_col
    for k in range(group_len):
        col = from_col + k
        if last_position[col] != 0:
            group_value += last_position[col] * last_val_price[col]
    return group_value
```

## get_group_value_ctx_nb
从上下文获取资产组价值，`get_group_value_nb` 的上下文版本。
- 自动从 `SegmentContext` 中提取所需的参数来计算资产组价值。

参数
- `seg_ctx`：SegmentContext，分段上下文对象，包含
  - `from_col, to_col`：当前资产组的列范围
  - `group`：当前资产组索引
  - `last_cash`：各组的最新现金数组
  - `last_position`：最新持仓数组
  - `last_val_price`：最新估值价格数组
  - `cash_sharing`：现金共享标志
    
返回：float，当前资产组的总价值

```python
@njit(cache=True)
def get_group_value_ctx_nb(seg_ctx: SegmentContext) -> float:
    if not seg_ctx.cash_sharing:
        raise ValueError("Cash sharing must be enabled")
    return get_group_value_nb(
        seg_ctx.from_col,
        seg_ctx.to_col,
        seg_ctx.last_cash[seg_ctx.group],
        seg_ctx.last_position,
        seg_ctx.last_val_price
    )
```

## approx_order_value_nb
根据订单参数和当前状态，估算订单的价值（所需现金量）。

参数
- `size`：float，订单大小，含义取决于 `size_type`
- `size_type`：int，订单大小类型，参见 `SizeType` 枚举：
  - `Amount`：具体数量。返回 `size * val_price_now`
  - `Value`：价值金额。返回 `size`
  - `Percent`：百分比（现金或持仓的百分比）
    - 如果 `size >= 0`（此时 `size` 表示占据当前现金余额的百分比）
      - 返回 `size * cash_now`
    - 否则
      - 如果 `direction == Direction.LongOnly`（此时 `size` 表示占据当前该资产价值的百分比）
        - 返回 `size * position_now * val_price_now`
      - 否则
        - 返回 `size * (2 * max(position_now * val_price_now, 0) + max(free_cash_now, 0))`
  - `TargetAmount`：目标持仓数量。
    - 此时 `size` 表示：加上该订单后，持有的该资产总数
    - 返回 `size * val_price_now - position_now * val_price_now`
  - `TargetValue`：目标持仓价值。
    - 此时 `size` 表示：加上该订单后，持有的该资产总价值
    - 返回 `size - position_now * val_price_now`
  - `TargetPercent`：目标持仓占组合的百分比。
    - 返回 `size * value_now - position_now * val_price_now`
- `direction`：int，交易方向限制，参见 `Direction` 枚举：
  - `LongOnly`：int = 0，仅多头：只允许多头（买入持有）仓位
  - `ShortOnly`：int = 1，仅空头：只允许空头（卖空）仓位  
  - `Both`：int = 2，双向：允许多头和空头仓位
- `cash_now`：float，当前现金余额
- `position_now`：float，当前持仓数量
- `free_cash_now`：float，当前可用现金
- `val_price_now`：float，当前估值价格
- `value_now`：float，当前投资组合总价值
  
返回：float， 估算的订单价值（所需现金）
- 正值表示买入，负值表示卖出，如果无法计算则返回 NaN

```python
@njit(cache=True)
def approx_order_value_nb(size: float,
                          size_type: int,
                          direction: int,
                          cash_now: float,
                          position_now: float,
                          free_cash_now: float,
                          val_price_now: float,
                          value_now: float) -> float:
    if direction == Direction.ShortOnly:
        size *= -1
    asset_value_now = position_now * val_price_now
    if size_type == SizeType.Amount:
        return size * val_price_now
    if size_type == SizeType.Value:
        return size
    if size_type == SizeType.Percent:
        if size >= 0:
            return size * cash_now
        else:
            if direction == Direction.LongOnly:
                return size * asset_value_now
            return size * (2 * max(asset_value_now, 0) + max(free_cash_now, 0))
    if size_type == SizeType.TargetAmount:
        return size * val_price_now - asset_value_now
    if size_type == SizeType.TargetValue:
        return size - asset_value_now
    if size_type == SizeType.TargetPercent:
        return size * value_now - asset_value_now
    return np.nan
```

## sort_call_seq_out_nb
基于订单价值对调用序列 `call_seq_out` 进行就地排序。

参数
- `ctx`：`SegmentContext`，分段上下文对象，包含当前状态信息
  - `target_shape`：tp.Shape，模拟目标形状
  - `group_lens`：tp.Array1d，每组列数
  - `init_cash`：tp.Array1d，初始资金
  - `cash_sharing`：bool，现金共享标志
  - `call_seq`：tp.Optional[tp.Array2d]，调用序列
  - `segment_mask`：tp.ArrayLike，段掩码
  - `call_pre_segment`：bool，调用段前函数标志
  - `call_post_segment`：bool，调用段后函数标志
  - `close`：tp.ArrayLike，收盘价数据
  - `ffill_val_price`：bool，前向填充估值价格标志
  - `update_value`：bool，更新价值标志
  - `fill_pos_record`：bool，填充仓位记录标志
  - `flex_2d`：bool，灵活二维索引标志
  - `order_records`：tp.RecordArray，订单记录数组
  - `log_records`：tp.RecordArray，日志记录数组
  - `last_cash`：tp.Array1d，最新现金状态
  - `last_position`：tp.Array1d，最新仓位状态
  - `last_debt`：tp.Array1d，最新债务状态
  - `last_free_cash`：tp.Array1d，最新可用现金
  - `last_val_price`：tp.Array1d，最新估值价格
  - `last_value`：tp.Array1d，最新组合价值
  - `second_last_value`：tp.Array1d，次新组合价值
  - `last_return`：tp.Array1d，最新收益率
  - `last_oidx`：tp.Array1d，最新订单索引
  - `last_lidx`：tp.Array1d，最新日志索引
  - `last_pos_record`：tp.RecordArray，最新仓位记录
  - `group`：int，当前组索引
  - `group_len`：int，当前组大小
  - `from_col`：int，组起始列索引
  - `to_col`：int，组结束列索引+1
  - `i`：int，当前行索引
  - `call_seq_now`：tp.Optional[tp.Array1d]，当前段内的调用序列
- `size`：ArrayLike，订单大小数组，支持灵活索引 
- `size_type`：ArrayLike，订单大小类型数组，支持灵活索引
- `direction`：ArrayLike，交易方向数组，支持灵活索引
- `order_value_out`：Array1d，输出的订单价值数组，长度应匹配组内资产数量
  - 函数执行前应为空，执行后包含排序后的订单价值
- `call_seq_out`：Array1d，输入/输出的调用序列数组，长度应匹配组内资产数量
  - 输入时应按默认顺序填充 (0, 1, 2, ...)；输出时为按价值排序后的资产索引序列
- `ctx_select`：bool, 可选 (默认: True)，索引选择模式
    - True：使用 `get_col_elem_nb` 进行上下文选择
    - False：使用 `flex_select_auto_nb` 进行灵活选择

```python
@njit(cache=True)
def sort_call_seq_out_nb(ctx: SegmentContext,
                         size: tp.ArrayLike,
                         size_type: tp.ArrayLike,
                         direction: tp.ArrayLike,
                         order_value_out: tp.Array1d,
                         call_seq_out: tp.Array1d,
                         ctx_select: bool = True) -> None:
    if not ctx.cash_sharing:
        raise ValueError("Cash sharing must be enabled")
    size_arr = np.asarray(size)
    size_type_arr = np.asarray(size_type)
    direction_arr = np.asarray(direction)

    # 获取当前组合总价值
    group_value_now = get_group_value_ctx_nb(ctx)
    group_len = ctx.to_col - ctx.from_col # 组内资产数量
    # 遍历组内每个资产，计算订单价值
    for k in range(group_len):
        if call_seq_out[k] != k:
            raise ValueError("call_seq_out should follow CallSeqType.Default")
        # 当前资产的绝对列索引
        col = ctx.from_col + k
        if ctx_select:
            _size = get_col_elem_nb(ctx, col, size_arr)
            _size_type = get_col_elem_nb(ctx, col, size_type_arr)
            _direction = get_col_elem_nb(ctx, col, direction_arr)
        else:
            _size = flex_select_auto_nb(size_arr, k, 0, False)
            _size_type = flex_select_auto_nb(size_type_arr, k, 0, False)
            _direction = flex_select_auto_nb(direction_arr, k, 0, False)
        # 获取现金状态（现金共享模式使用组级现金）
        if ctx.cash_sharing:
            cash_now = ctx.last_cash[ctx.group]
            free_cash_now = ctx.last_free_cash[ctx.group]
        else:
            cash_now = ctx.last_cash[col]
            free_cash_now = ctx.last_free_cash[col]
        # 计算该资产的近似订单价值
        order_value_out[k] = approx_order_value_nb(
            _size,
            _size_type,
            _direction,
            cash_now,
            ctx.last_position[col],
            free_cash_now,
            ctx.last_val_price[col],
            group_value_now
        )
    # 根据订单价值对调用序列进行排序
    # 使用插入排序算法，按价值从大到小排序
    insert_argsort_nb(order_value_out, call_seq_out)
```

## sort_call_seq_nb
`sort_call_seq_out_nb` 的简化版本，直接对上下文中的当前调用序列 `ctx.call_seq_now` 进行排序，无需提供额外的 call_seq_out 参数。

参数
- `ctx`：SegmentContext，分段上下文对象，必须包含有效的 `call_seq_now` 属性 
- `size`：ArrayLike，订单大小数组，支持灵活索引
- `size_type`：ArrayLike，订单大小类型数组，支持灵活索引
- `direction`：ArrayLike，交易方向数组，支持灵活索引
- `order_value_out`：Array1d，输出的订单价值数组，长度应匹配组内资产数量
- `ctx_select`：bool, 可选 (默认: True)，索引选择模式，参见 `sort_call_seq_out_nb`

```python
@njit(cache=True)
def sort_call_seq_nb(ctx: SegmentContext,
                     size: tp.ArrayLike,
                     size_type: tp.ArrayLike,
                     direction: tp.ArrayLike,
                     order_value_out: tp.Array1d,
                     ctx_select: bool = True) -> None:
    if ctx.call_seq_now is None:
        raise ValueError("Call sequence array is None. Use sort_call_seq_out_nb to sort a custom array.")
    sort_call_seq_out_nb(
        ctx,
        size,
        size_type,
        direction,
        order_value_out,
        ctx.call_seq_now,
        ctx_select=ctx_select
    )
```

## replace_inf_price_nb
替换订单中的无穷价格：将订单中的无穷价格（np.inf）替换为实际的市场价格。

参数
- `prev_close`：float，上一个收盘价，用作价格下限
- `close`：float，当前收盘价，用作价格上限
- `order`：Order，包含无穷价格的原始订单对象
    
返回：Order，价格已替换的新订单对象
    
价格替换规则：
- 如果订单价格 `order.price > 0`：使用当前收盘价作为上限（买入市价单）
- 如果订单价格 `order.price <= 0`：使用前收盘价作为下限（卖出市价单）

```python
@njit(cache=True)
def replace_inf_price_nb(prev_close: float, close: float, order: Order) -> Order:
    order_price = order.price
    if order_price > 0:
        # 正价格（通常是买入市价单）：使用当前收盘价作为上限
        order_price = close  # upper bound is close
    else:
        # 负价格或零（通常是卖出市价单）：使用前收盘价作为下限
        order_price = prev_close  # lower bound is prev close
    return order_nb(
        size=order.size,
        price=order_price,  # 替换后的价格
        size_type=order.size_type,
        direction=order.direction,
        fees=order.fees,
        fixed_fees=order.fixed_fees,
        slippage=order.slippage,
        min_size=order.min_size,
        max_size=order.max_size,
        size_granularity=order.size_granularity,
        reject_prob=order.reject_prob,
        lock_cash=order.lock_cash,
        allow_partial=order.allow_partial,
        raise_reject=order.raise_reject,
        log=order.log
    )
```

## try_order_nb
根据上下文对象 `ctx` 创建临时状态 `ProcessOrderState` 来尝试执行订单 `order`。

参数
- `ctx`：OrderContext，订单上下文对象，包含当前的投资组合状态
  - `cash_now`：当前现金
  - `position_now`：当前持仓
  - `debt_now`：当前债务
  - `free_cash_now`：当前可用现金
  - `val_price_now`：当前估值价格
  - `value_now`：当前投资组合价值
  - `i`：当前时间索引
  - `col`：当前资产列索引
  - `close`：收盘价数组
- `order`：Order，待测试执行的订单对象
    
返回：`tuple[ExecuteOrderState, OrderResult]` (执行后状态, 订单结果) 。

```python
@njit(cache=True)
def try_order_nb(ctx: OrderContext, order: Order) -> tp.Tuple[ExecuteOrderState, OrderResult]:
    # 从上下文 ctx 创建临时的处理状态，记录索引设为 -1 表示这是测试执行
    state = ProcessOrderState(
        cash=ctx.cash_now,
        position=ctx.position_now,
        debt=ctx.debt_now,
        free_cash=ctx.free_cash_now,
        val_price=ctx.val_price_now,
        value=ctx.value_now,
        oidx=-1,
        lidx=-1
    )
    # 处理无穷价格（市价订单）
    if np.isinf(order.price):
        # 获取前一个收盘价（如果存在）
        if ctx.i > 0:
            prev_close = flex_select_auto_nb(ctx.close, ctx.i - 1, ctx.col, ctx.flex_2d)
        else:
            prev_close = np.nan
        # 获取当前收盘价
        close = flex_select_auto_nb(ctx.close, ctx.i, ctx.col, ctx.flex_2d)
        # 将无穷价格替换为实际市场价格
        order = replace_inf_price_nb(prev_close, close, order)
    # 执行订单并返回结果（不持久化状态）
    return execute_order_nb(state, order)
```

## init_records_nb
初始化订单和日志记录数组。

参数
- `target_shape`：Shape，目标形状 (时间步数, 资产数)
- `max_orders`：int，可选 (默认: None)，最大订单记录数量
  - None：自动计算为时间步数 × 资产数 (每个位置最多一个订单)
  - 具体数值：使用指定的容量限制
- `max_logs`：int, 可选 (默认: 0)，最大日志记录数量：
  - `0`: 创建最小容量（1个记录）
  - `>0`: 使用指定的容量
    
返回：`tuple[RecordArray, RecordArray]` (订单记录数组, 日志记录数组) 
- 已初始化但为空的记录容器

```python
@njit(cache=True)
def init_records_nb(target_shape: tp.Shape,
                    max_orders: tp.Optional[int] = None,
                    max_logs: int = 0) -> tp.Tuple[tp.RecordArray, tp.RecordArray]:
    # 计算订单记录的最大数量
    if max_orders is None:
        _max_orders = target_shape[0] * target_shape[1]
    else:
        _max_orders = max_orders
    # 创建空的订单记录数组
    order_records = np.empty(_max_orders, dtype=order_dt)
    # 处理日志记录容量
    if max_logs == 0:
        max_logs = 1
    # 创建空的日志记录数组
    log_records = np.empty(max_logs, dtype=log_dt)
    return order_records, log_records
```

## update_open_pos_stats_nb
根据 `record` 将出场价格更新为 $\frac{{\left( {总数量 - 出场数量} \right) \cdot 出场价格 + 剩余数量 \cdot 当前价格}}{总数量}$，然后计算并更新 `record` 的盈亏 `pnl` 和收益率 `return`。

参数
- `record`：Record，头寸记录对象，包含入场价格、手续费等信息
- `position_now `：float，当前剩余头寸数量（绝对值） 
- `price`：float，用于计算的当前价格（通常是最新市价）
    
就地修改：更新 `record` 中的 `pnl` 和 `return` 字段。

```python
@njit(cache=True)
def update_open_pos_stats_nb(record: tp.Record, position_now: float, price: float) -> None:
    # 检查记录是否为有效的开仓状态
    if record['id'] >= 0 and record['status'] == TradeStatus.Open:
        # 处理出场价格的计算
        if np.isnan(record['exit_price']):
            # 如果尚未设置出场价格，直接使用当前价格
            exit_price = price
        else:
            # 如果已有出场价格（部分平仓），计算加权平均价格
            exit_size_sum = record['size'] - abs(position_now)
            exit_gross_sum = exit_size_sum * record['exit_price']
            exit_gross_sum += abs(position_now) * price
            exit_price = exit_gross_sum / record['size']
        # 计算盈亏和收益率
        pnl, ret = get_trade_stats_nb(
            record['size'],
            record['entry_price'],
            record['entry_fees'],
            exit_price,
            record['exit_fees'],
            record['direction']
        )
        record['pnl'] = pnl
        record['return'] = ret
```

## update_pos_record_nb
根据订单执行结果 `order_result：OrderResult` 更新头寸记录对象 `record：Record`。

参数
- `record`：Record，头寸记录对象，存储交易的详细信息
  - `id`：交易ID
  - `col`：资产列索引
  - `size`：头寸大小
  - `entry_idx`：入场时间索引
  - `entry_price`：入场价格
  - `entry_fees`：入场手续费
  - `exit_idx`：出场时间索引
  - `exit_price`：出场价格
  - `exit_fees`：出场手续费
  - `direction`：交易方向
  - `status`：交易状态
  - `parent_id`：父交易 ID
- `i`：int，当前时间索引（时间步）
- `col`：int，资产列索引（资产标识符）
- `position_before`：float，订单执行前的头寸数量（可为正负）
- `position_now`：float，订单执行后的头寸数量（可为正负）
- `order_result`：OrderResult，订单执行结果对象
    
就地修改：更新 `record` 中的各项字段，反映最新的交易状态和统计信息。

逻辑：
1. *开仓 (position_before=0 → position_now≠0)*
   - 根据参数设置交易记录 `record`
   - 设置状态为开仓即 `record['status'] = TradeStatus.Open`
2. *平仓 (position_before≠0 → position_now=0)*
   - 更新出场价格 `record['exit_price']`
     - $$\frac{{\left( {原始买入数量 - 当前持有数量} \right) \cdot 之前出场价格 + 当前持有数量 \cdot 当前出场价格}}{原始买入数量}$$
   - 更新出场手续费 `record['exit_fees'] += order_result.fees`
   - 使用 `get_trade_stats_nb` 计算 `record` 的最终盈亏和收益率
   - 设置状态为关闭即 `record['status'] = TradeStatus.Closed `
3. *反转 (position_before与position_now符号相反)*
   - 重置 `entry_fees` $= \frac{{\left| position\_now \right|}}{{\left| {position\_now - position\_before} \right|}}order\_result.fees$，以及 `exit_fees` $=0$
   - 设置状态为开仓即 `record['status'] = TradeStatus.Open`
4. *调整 (同方向但大小变化)*
   - *增仓（|position_before| ≤ |position_now|）*
     - 更新入场价格 `entry_Price`$= \frac{{原始数量 \cdot 之前入场价格 + order\_result.size \cdot order\_result.price}}{原始数量+order\_result.size}$
     - 更新入场手续费：`record['entry_fees'] += order_result.fees`
     - 更新总头寸：`record['size'] += order_result.size`
   - *减仓（|position_before| > |position_now|）*
     - 更新出场价格 `exit_price`$= \frac{{\left( {原始买入数量 - \left| {position\_before} \right|} \right) \cdot 之前出场价格 + order\_result.size \cdot order\_result.price}}{原始买入数量 - \left| {position\_before} \right|+order\_result.size}$
     - 更新出场手续费：`record['exit_fees'] += order_result.fees`
- 使用 `update_open_pos_stats_nb` 更新 `record` 的盈亏和收益率

```python
@njit(cache=True)
def update_pos_record_nb(record: tp.Record,
                         i: int,
                         col: int,
                         position_before: float,
                         position_now: float,
                         order_result: OrderResult) -> None:
    # 只处理成功执行的订单
    if order_result.status == OrderStatus.Filled:
        # 情况1：开新仓位（从零头寸变为非零头寸，position_before=0 → position_now≠0）
        if position_before == 0 and position_now != 0:
            # 创建新交易记录
            record['id'] += 1                           # 递增交易ID
            record['col'] = col                         # 资产列索引
            record['size'] = order_result.size          # 交易数量
            record['entry_idx'] = i                     # 入场时间索引
            record['entry_price'] = order_result.price  # 入场价格
            record['entry_fees'] = order_result.fees    # 入场手续费
            record['exit_idx'] = -1                     # 出场时间（未定）
            record['exit_price'] = np.nan               # 出场价格（未定）
            record['exit_fees'] = 0.                    # 出场手续费（初始为0）
            
            # 根据订单方向确定交易方向
            if order_result.side == OrderSide.Buy:
                record['direction'] = TradeDirection.Long   # 买入为多头
            else:
                record['direction'] = TradeDirection.Short  # 卖出为空头
                
            record['status'] = TradeStatus.Open         # 状态为开仓
            record['parent_id'] = record['id']          # 父交易ID（自引用）
        # 情况2：平仓（从非零头寸变为零头寸，position_before≠0 → position_now=0）
        elif position_before != 0 and position_now == 0:
            # 完成交易记录
            record['exit_idx'] = i                      # 出场时间索引
            
            # 计算加权平均出场价格
            if np.isnan(record['exit_price']):
                # 首次出场，直接使用当前价格
                exit_price = order_result.price
            else:
                # 多次出场，计算加权平均价格
                exit_size_sum = record['size'] - abs(position_before)  # 之前出场的数量
                exit_gross_sum = exit_size_sum * record['exit_price']  # 之前出场的总价值
                exit_gross_sum += abs(position_before) * order_result.price  # 加上本次出场价值
                exit_price = exit_gross_sum / record['size']  # 加权平均出场价格
                
            record['exit_price'] = exit_price           # 更新出场价格
            record['exit_fees'] += order_result.fees    # 累加出场手续费
            
            # 计算最终的盈亏和收益率
            pnl, ret = get_trade_stats_nb(
                record['size'],         # 交易数量
                record['entry_price'],  # 入场价格
                record['entry_fees'],   # 入场手续费
                record['exit_price'],   # 出场价格
                record['exit_fees'],    # 出场手续费
                record['direction']     # 交易方向
            )
            record['pnl'] = pnl                        # 净盈亏
            record['return'] = ret                     # 收益率
            record['status'] = TradeStatus.Closed      # 状态为已关闭
        # 情况3：头寸反转（正负号改变，position_before 与 position_now 符号相反）
        elif np.sign(position_before) != np.sign(position_now):
            # 生成新的交易记录
            record['id'] += 1                          # 递增交易ID
            record['size'] = abs(position_now)         # 新头寸的绝对数量
            record['entry_idx'] = i                    # 新入场时间
            record['entry_price'] = order_result.price # 新入场价格
            
            # 按新头寸比例分配手续费
            new_pos_fraction = abs(position_now) / abs(position_now - position_before)
            record['entry_fees'] = new_pos_fraction * order_result.fees
            
            # 重置出场信息
            record['exit_idx'] = -1
            record['exit_price'] = np.nan
            record['exit_fees'] = 0.
            
            # 确定新的交易方向
            if order_result.side == OrderSide.Buy:
                record['direction'] = TradeDirection.Long
            else:
                record['direction'] = TradeDirection.Short
                
            record['status'] = TradeStatus.Open        # 状态为开仓
            record['parent_id'] = record['id']         # 新的父交易ID
        # 情况4：头寸调整（同方向但大小变化）
        else:
            if abs(position_before) <= abs(position_now):
                # 增仓：计算加权平均入场价格
                entry_gross_sum = record['size'] * record['entry_price']  # 原入场总价值
                entry_gross_sum += order_result.size * order_result.price  # 加上新入场价值
                entry_price = entry_gross_sum / (record['size'] + order_result.size)  # 加权平均
                
                record['entry_price'] = entry_price    # 更新入场价格
                record['entry_fees'] += order_result.fees  # 累加入场手续费
                record['size'] += order_result.size    # 更新总头寸大小
                
            else:
                # 减仓：计算加权平均出场价格
                if np.isnan(record['exit_price']):
                    # 首次减仓
                    exit_price = order_result.price
                else:
                    # 多次减仓，计算加权平均价格
                    exit_size_sum = record['size'] - abs(position_before)  # 之前减仓数量
                    exit_gross_sum = exit_size_sum * record['exit_price']  # 之前减仓价值
                    exit_gross_sum += order_result.size * order_result.price  # 本次减仓价值
                    exit_price = exit_gross_sum / (exit_size_sum + order_result.size)  # 加权平均
                    
                record['exit_price'] = exit_price      # 更新出场价格
                record['exit_fees'] += order_result.fees  # 累加出场手续费

        # 更新开仓头寸的统计信息
        update_open_pos_stats_nb(
            record,                 # 头寸记录
            position_now,           # 当前头寸数量
            order_result.price      # 当前价格
        )
```

# 投资组合模拟

## simulate_from_orders_nb
基于订单矩阵的投资组合模拟。
- `price` 为*时间步×资产*。其中资产进一步分为几块，分别对应不同的投资组合。
- 根据 `price` 构造订单，构造的循环顺序为 *`分组——>时间步——>分组中的资产`*，基本处理方式为 `process_order_nb`。

### 参数和返回
- `target_shape`：Shape，目标形状 (时间步数, 资产数)，定义回测的时间和资产维度
- `group_lens`：Array1d，各组的长度数组，用于资产分组管理
- `init_cash`：Array1d，各组或各资产的初始现金数组
- `call_seq`：Array2d，调用序列数组，控制资产的处理顺序
- `size`：ArrayLike, 可选 (默认: np.inf)，订单大小数组，支持灵活广播
  - np.inf: 使用全部可用现金
  - 具体数值: 按指定数量或比例交易 
- `price`：ArrayLike, 可选 (默认: np.inf)，订单价格数组，支持灵活广播
    - np.inf: 市价订单，使用收盘价
    - 具体数值: 限价订单
- `size_type`：ArrayLike, 可选 (默认: SizeType.Amount)，订单大小类型，控制 `size` 参数的解释方式
- `direction`：ArrayLike, 可选 (默认: Direction.Both)，交易方向限制
- `fees`：ArrayLike, 可选 (默认: 0.0)，比例手续费率数组
- `fixed_fees`：ArrayLike, 可选 (默认: 0.0)，固定手续费数组
- `slippage`：ArrayLike, 可选 (默认: 0.0)，滑点率数组
- `min_size`：ArrayLike, 可选 (默认: 0.0)，最小订单大小数组
- `max_size`：ArrayLike, 可选 (默认: np.inf)，最大订单大小数组
- `size_granularity`：ArrayLike, 可选 (默认: np.nan)，订单数量粒度数组
- `reject_prob`：ArrayLike, 可选 (默认: 0.0)，随机拒绝概率数组，用于压力测试
- `lock_cash`：ArrayLike, 可选 (默认: False)，现金锁定标志数组
- `allow_partial`：ArrayLike, 可选 (默认: True)，允许部分成交标志数组
- `raise_reject`：ArrayLike, 可选 (默认: False)，拒绝时抛异常标志数组
- `log`：ArrayLike, 可选 (默认: False)，日志记录标志数组
- `val_price`：ArrayLike, 可选 (默认: np.inf)，估值价格数组，用于组合价值计算
- `close`：ArrayLike, 可选 (默认: np.nan)，收盘价数组，用于市价订单和估值
- `auto_call_seq`：bool, 可选 (默认: False)，是否自动按订单价值排序调用序列
- `ffill_val_price`：bool, 可选 (默认: True)，是否前向填充估值价格
- `update_value`：bool, 可选 (默认: False)，是否在每次订单后更新组合价值
- `max_orders`：int, 可选 (默认: None)，最大订单记录数，None时自动计算
- `max_logs`：int, 可选 (默认: 0)，最大日志记录数
- `flex_2d`：bool, 可选 (默认: True),是否启用2D灵活索引
    
返回：`tuple[RecordArray, RecordArray]` (订单记录数组, 日志记录数组)

### 逻辑

- 验证
  - 分组包括了所有资产：分组序列 `group_lens` 各元素的和要与 `target_shape[1]` 相同
  - 是否真正具有非平凡的分组 `cash_sharing = np.any(group_lens > 1)`
  - 现金配置与分组的兼容性
    - `cash_sharing` 即现金在组内共享时，`init_cash` 和 `group_lens` 的长度须相同
    - `!cash_sharing` 即非现金共享时，`init_cash` 的长度必须是资产数 `target_shape[1]`
- 初始化
  - `last_position`，`last_debt`，`last_val_price`，`order_price`，`temp_order_value`
- 下面的循环嵌套结构为 *`分组——>时间步——>分组中的资产`*
- *循环处理每个分组 `for group in range(len(group_lens)):`*
  - 初始化组级现金状态 `cash_now = init_cash[group]`，`free_cash_now = init_cash[group]`
  - *循环处理每个时间步 `for i in range(target_shape[0]):`*
    - 1.解析组内每个资产的订单价格和估值价格：
      - 对于 `price[i, col]`，根据下述情况缓存到 `order_price[col]`：
        - +inf ——> close[i, col]
        - -inf 并且 i > 0 ——> close[i-1, col]
        - -inf 并且 i= 0 ——> NaN
        - 否则，保持
      - 对于 `val_price[i, col]`，如果其不是 NaN 且 `!ffill_val_price`，根据下述情况缓存到 `last_val_price[col]`：
        - +inf ——> order_price[col]
        - -inf 并且 i > 0 ——> close[i-1, col]
        - -inf 并且 i= 0 ——> NaN
        - 否则，保持
    - 2.组价值计算以及订单排序
      - 如果现金共享 `cash_sharing`，计算组的总价值 `value_now`；
        - `value_now = cash_now + last_position[from_col] * last_val_price[from_col] + ...`
      - 如果 `auto_call_seq` 即按订单价值排序
        - 使用 `temp_order_value[k] = approx_order_value_nb` 计算每个资产的价值，缓存到 `temp_order_value`
        - 按照价值排序，将顺序存储到 `call_seq[i, from_col:to_col]`
    - 3.*处理组内每个资产的订单 for k in range(group_len):*
      - `col = from_col + k`。如果 `cash_sharing` 即现金共享，`col = from_col + call_seq[i, col]`
      - 获取当前资产状态
        - 当前持仓：`position_now = last_position[col]`
        - 当前债务：`debt_now = last_debt[col]`
        - 当前估值价格：`val_price_now = last_val_price[col]`
      - 非现金共享模式为每个资产单独计算价值 `value_now`
        - `value_now = cash_now + position_now * val_price_now`
      - 使用 `order_nb` 创建订单对象 `order`
      - 根据当前组状态创建 `state: ProcessOrderState`
      - 使用 `process_order_nb` 执行订单 `order`
        - 会更新订单记录 `order_records` 和日志记录 `log_records`
      - 更新组级别状态 `state`
      - 更新资产级别状态 `last_position`，`last_debt`，`last_val_price`
- 返回订单记录 `order_records[:oidx]` 和日志记录 `log_records[:lidx]`

## generate_stop_signal_nb
根据当前持仓 `position_now` 以及止损退出模式 `upon_stop_exit`，生成止损信号。
- 可能调整累积模式，即参数 `accumulate`
- 止损信号为 `tuple[bool, bool, bool, bool, int]`  
  - 对应 `(is_long_entry, is_long_exit, is_short_entry, is_short_exit, new_accumulate)`
  - `is_long_entry`：是否生成多头开仓信号
  - `is_long_exit`：是否生成多头平仓信号
  - `is_short_entry`：是否生成空头开仓信号
  - `is_short_exit`：是否生成空头平仓信号
  - `new_accumulate`：调整后的累积模式

参数
- `position_now`：float，当前持仓数量（正数为多头，负数为空头，0为无持仓）
- `upon_stop_exit`：int，止损退出模式，参见 `StopExitMode` 枚举：
  - `Close`：完全关闭持仓
  - `CloseReduce`：关闭或减少持仓 
  - `Reverse`： 反转持仓方向
  - `ReverseReduce`：反转或减少持仓
- `accumulate`：int，当前累积模式，可能被此函数修改

### 源码
```python
@njit(cache=True)
def generate_stop_signal_nb(position_now: float,
                            upon_stop_exit: int,
                            accumulate: int) -> tp.Tuple[bool, bool, bool, bool, int]:
    is_long_entry = False
    is_long_exit = False
    is_short_entry = False
    is_short_exit = False
    if position_now > 0:
        if upon_stop_exit == StopExitMode.Close:
            is_long_exit = True
            accumulate = AccumulationMode.Disabled
        elif upon_stop_exit == StopExitMode.CloseReduce:
            is_long_exit = True
        elif upon_stop_exit == StopExitMode.Reverse:
            is_short_entry = True
            accumulate = AccumulationMode.Disabled
        else:
            is_short_entry = True
    elif position_now < 0:
        if upon_stop_exit == StopExitMode.Close:
            is_short_exit = True
            accumulate = AccumulationMode.Disabled
        elif upon_stop_exit == StopExitMode.CloseReduce:
            is_short_exit = True
        elif upon_stop_exit == StopExitMode.Reverse:
            is_long_entry = True
            accumulate = AccumulationMode.Disabled
        else:
            is_long_entry = True
    return is_long_entry, is_long_exit, is_short_entry, is_short_exit, accumulate
```

## resolve_stop_price_and_slippage_nb
根据指定的止损退出价格模式 `stop_exit_price`，确定最终的订单执行价格和滑点参数。

参数
- `stop_price`：float，原始止损触发价格
- `price`：float，当前市场价格
- `close`：float，收盘价格
- `slippage`：float，原始滑点设置
- `stop_exit_price`：int，止损退出价格模式，参见 `StopExitPrice` 枚举：
  - `StopMarket`：以止损价格执行市价单
  - `StopLimit`：以止损价格执行限价单
  - `Close`：以收盘价执行
  - `Price`：以当前价格执行
    
返回：tuple[float, float]，最终的执行价格和滑点

### 源码
```python
@njit(cache=True)
def resolve_stop_price_and_slippage_nb(stop_price: float,
                                       price: float,
                                       close: float,
                                       slippage: float,
                                       stop_exit_price: int) -> tp.Tuple[float, float]:
    if stop_exit_price == StopExitPrice.StopMarket:
        return stop_price, slippage
    elif stop_exit_price == StopExitPrice.StopLimit:
        return stop_price, 0.
    elif stop_exit_price == StopExitPrice.Close:
        return close, slippage
    return price, slippage
```

## resolve_signal_conflict_nb
同时出现开仓和平仓信号时，根据指定的冲突处理模式，决定最终执行哪个信号。
### 参数和返回
- `position_now`：float，当前持仓数量（正数为多头，负数为空头，0为无持仓）
    
- `is_entry`：bool，是否有开仓信号
    
- `is_exit`：bool，是否有平仓信号
    
- `direction`：int，交易方向限制，参见 `Direction` 枚举
  - `LongOnly`：int = 0，仅多头：只允许多头（买入持有）仓位
  - `ShortOnly`：int = 1，仅空头：只允许空头（卖空）仓位  
  - `Both`：int = 2，双向：允许多头和空头仓位
- `conflict_mode`：int，冲突处理模式，参见 `ConflictMode` 枚举：
  - `Entry`：优先开仓信号，忽略平仓信号
  - `Exit`：优先平仓信号，忽略开仓信号  
  - `Adjacent`：选择与当前持仓"相邻"的信号
  - `Opposite`：选择与当前持仓"相反"的信号
  - `Ignore`：忽略所有冲突信号
    
返回：tuple[bool, bool] (final_is_entry, final_is_exit) —— 解决冲突后的最终信号

### 逻辑
- 如果同时存在开仓信号 `is_entry` 和平仓信号 `is_exit`，根据冲突处理模式 `conflict_mode`
  - 优先开仓 `ConflictMode.Entry`
    - 忽略平仓信号 `is_exit = False`
  - 优先平仓 `ConflictMode.Exit`
    - 忽略开仓信号 `is_entry = False`
  - 相邻模式 `ConflictMode.Adjacent`
    - 当前无头寸 `position_now == 0`
      - 全部忽略 `is_exit = is_entry = False`
    - 否则
      - 如果交易方向为双向 `direction == Direction.Both`
        - 如果当前多头持仓 `position_now > 0`
          - 只保留开仓信号 `is_exit = False`（继续做多）
        - 如果当前空头持仓 `position_now < 0`
          - 只保留平仓信号 `is_entry = False`（平空头）
      - 否则（即单向交易）
        - 总是只保留开仓信号 `is_exit = False`
  - 相反模式 `ConflictMode.Opposite`
    - 当前无头寸 `position_now == 0`
      - 全部忽略 `is_exit = is_entry = False`
    - 否则
      - 如果交易方向为双向 `direction == Direction.Both`
        - 如果当前多头持仓 `position_now > 0`
          - 只保留平仓信号 `is_entry = False`（平多头）
        - 如果当前空头持仓 `position_now < 0`
          - 只保留开仓信号 `is_exit = False`（继续做空）
      - 否则（即单向交易）
        - 总是只保留开仓信号 `is_exit = False`
  - 忽略模式 `ConflictMode.Ignore`
    - 全部忽略 `is_exit = is_entry = False`

### 源码
```python
@njit(cache=True)
def resolve_signal_conflict_nb(position_now: float,
                               is_entry: bool,
                               is_exit: bool,
                               direction: int,
                               conflict_mode: int) -> tp.Tuple[bool, bool]:
    if is_entry and is_exit:
        # Conflict
        if conflict_mode == ConflictMode.Entry:
            is_exit = False
        elif conflict_mode == ConflictMode.Exit:
            is_entry = False
        elif conflict_mode == ConflictMode.Adjacent:
            if position_now == 0:
                # Cannot decide -> ignore
                is_entry = False
                is_exit = False
            else:
                if direction == Direction.Both:
                    if position_now > 0:
                        is_exit = False
                    elif position_now < 0:
                        is_entry = False
                else:
                    is_exit = False
        elif conflict_mode == ConflictMode.Opposite:
            if position_now == 0:
                is_entry = False
                is_exit = False
            else:
                if direction == Direction.Both:
                    if position_now > 0:
                        is_entry = False
                    elif position_now < 0:
                        is_exit = False
                else:
                    is_entry = False
        else:
            is_entry = False
            is_exit = False
    return is_entry, is_exit
```

## resolve_dir_conflict_nb
同时出现多头加仓和空头加仓信号时，根据指定的冲突处理模式，决定最终执行哪个信号。

参数
- `position_now`：float，当前持仓数量（正数为多头，负数为空头，0为无持仓）
- `is_long_entry`：bool，是否有多头开仓信号
- `is_short_entry`：bool，是否有空头开仓信号
- `upon_dir_conflict`：int，方向冲突处理模式，参见 `DirectionConflictMode` 枚举：
  - `Long`：优先多头开仓信号
  - `Short`：优先空头开仓信号
  - `Adjacent`：选择与当前持仓方向一致的信号
  - `Opposite`：选择与当前持仓方向相反的信号
  - `Ignore`：忽略所有冲突信号
    
返回：tuple[bool, bool] (final_is_long_entry, final_is_short_entry) —— 解决冲突后的最终信号

### 逻辑
- 如果同时存在多头加仓信号 `is_long_entry` 和空头加仓信号 `is_short_entry`，根据冲突处理模式 `DirectionConflictMode`
  - 优先多头加仓 `DirectionConflictMode.Long`
    - 忽略空头加仓信号 `is_short_entry = False`
  - 优先空头加仓 `DirectionConflictMode.Short`
    - 忽略多头加仓信号 `is_long_entry = False`
  - 相邻模式 `DirectionConflictMode.Adjacent`
    - 当前无头寸 `position_now == 0`
      - 全部忽略 `is_long_entry = is_short_entry = False`
    - 当前多头持仓 `position_now > 0`
      - 只保留多头加仓信号 `is_short_entry = False`（顺势加仓）
    - 当前空头持仓 `position_now < 0`
      - 只保留空头加仓信号 `is_long_entry = False`（顺势加仓）
  - 相反模式 `DirectionConflictMode.Opposite`
    - 当前无头寸 `position_now == 0`
      - 全部忽略 `is_long_entry = is_short_entry = False`
    - 当前多头持仓 `position_now > 0`
      - 只保留空头加仓信号 `is_long_entry = False`（反转做空）
    - 当前空头持仓 `position_now < 0`
      - 只保留多头加仓信号 `is_short_entry = False`（反转做多）
  - 忽略模式 `ConflictMode.Ignore`
    - 全部忽略 `is_exit = is_entry = False`

### 源码
```python
@njit(cache=True)
def resolve_dir_conflict_nb(position_now: float,
                            is_long_entry: bool,
                            is_short_entry: bool,
                            upon_dir_conflict: int) -> tp.Tuple[bool, bool]:
    if is_long_entry and is_short_entry:
        if upon_dir_conflict == DirectionConflictMode.Long:
            is_short_entry = False
        elif upon_dir_conflict == DirectionConflictMode.Short:
            is_long_entry = False
        elif upon_dir_conflict == DirectionConflictMode.Adjacent:
            if position_now > 0:
                is_short_entry = False
            elif position_now < 0:
                is_long_entry = False
            else:
                is_long_entry = False
                is_short_entry = False
        elif upon_dir_conflict == DirectionConflictMode.Opposite:
            if position_now > 0:
                is_long_entry = False
            elif position_now < 0:
                is_short_entry = False
            else:
                is_long_entry = False
                is_short_entry = False
        else:
            is_long_entry = False
            is_short_entry = False
    return is_long_entry, is_short_entry
```

## resolve_opposite_entry_nb
持仓状况 `position_now` 与开仓方向相反的处理。

### 参数
- `position_now`：float，当前持仓数量（正数为多头，负数为空头，0为无持仓）
- `is_long_entry`：bool，是否有多头开仓信号
- `is_long_exit`：bool，是否有多头平仓信号
- `is_short_entry`：bool，是否有空头开仓信号
- `is_short_exit`：bool，是否有空头平仓信号
- `upon_opposite_entry`：int，相反开仓处理模式，参见 `OppositeEntryMode` 枚举：
  - `Ignore`：忽略相反开仓信号
  - `Close`：转换为平仓信号，禁用累积
  - `CloseReduce`：转换为平仓信号，保持累积
  - `Reverse`：允许反转，禁用累积
  - `ReverseReduce`：允许反转，保持累积
- `accumulate`：int，当前累积模式，可能被此函数修改
    
返回：tuple[bool, bool, bool, bool, int]，(is_long_entry, is_long_exit, is_short_entry, is_short_exit, new_accumulate)
- 调整后的信号状态和累积模式

### 逻辑
- 多头持仓 `position_now > 0` 时处理空头信号 `is_short_entry`，根据反向开仓模式 `upon_opposite_entry`
  - 忽略模式 `OppositeEntryMode.Ignore`
    - 忽略空头开仓信号 `is_short_entry = False`
  - 平仓模式 `OppositeEntryMode.Close`
    - 取消空头开仓，生成多头平仓，且禁用累积
      - `is_short_entry = False`
      - `is_long_exit = True`
      - `accumulate = AccumulationMode.Disabled`
  - 平仓减少模式 `OppositeEntryMode.CloseReduce`
    - 取消空头开仓，生成多头平仓，保持累积
      - `is_short_entry = False`
      - `is_long_exit = True`
  - 反转模式 `OppositeEntryMode.Reverse`
    - 保持，禁用累积
      - `is_short_entry` 保持为 `True`
      - `accumulate = AccumulationMode.Disabled` 
- 空头持仓 `position_now < 0` 时处理多头信号 `is_long_entry`，根据反向开仓模式 `upon_opposite_entry`
  - 忽略模式 `OppositeEntryMode.Ignore`
    - 忽略多头开仓信号 `is_long_entry = False`
  - 平仓模式 `OppositeEntryMode.Close`
    - 取消多头开仓，生成空头平仓，且禁用累积
      - `is_long_entry = False`
      - `is_short_exit = True`
      - `accumulate = AccumulationMode.Disabled`
  - 平仓减少模式 `OppositeEntryMode.CloseReduce`
    - 取消多头开仓，生成空头平仓，保持累积
      - `is_long_entry = False`
      - `is_short_exit = True`
  - 反转模式 `OppositeEntryMode.Reverse`
    - 保持，禁用累积
      - `is_long_entry` 保持为 `True`
      - `accumulate = AccumulationMode.Disabled` 

### 源码
```python
@njit(cache=True)
def resolve_opposite_entry_nb(position_now: float,
                              is_long_entry: bool,
                              is_long_exit: bool,
                              is_short_entry: bool,
                              is_short_exit: bool,
                              upon_opposite_entry: int,
                              accumulate: int) -> tp.Tuple[bool, bool, bool, bool, int]:
    if position_now > 0 and is_short_entry:
        if upon_opposite_entry == OppositeEntryMode.Ignore:
            is_short_entry = False
        elif upon_opposite_entry == OppositeEntryMode.Close:
            is_short_entry = False
            is_long_exit = True
            accumulate = AccumulationMode.Disabled
        elif upon_opposite_entry == OppositeEntryMode.CloseReduce:
            is_short_entry = False
            is_long_exit = True
        elif upon_opposite_entry == OppositeEntryMode.Reverse:
            accumulate = AccumulationMode.Disabled
    if position_now < 0 and is_long_entry:
        if upon_opposite_entry == OppositeEntryMode.Ignore:
            is_long_entry = False
        elif upon_opposite_entry == OppositeEntryMode.Close:
            is_long_entry = False
            is_short_exit = True
            accumulate = AccumulationMode.Disabled
        elif upon_opposite_entry == OppositeEntryMode.CloseReduce:
            is_long_entry = False
            is_short_exit = True
        elif upon_opposite_entry == OppositeEntryMode.Reverse:
            accumulate = AccumulationMode.Disabled
    return is_long_entry, is_long_exit, is_short_entry, is_short_exit, accumulate
```

## signals_to_size_nb
基于
- 持仓情况 `position_now`
- 订单大小 `size` 和订单类型 `size_type`
- 累积模式 `accumulate`
- 交易信号
  - 多头开仓 `is_long_entry`
  - 多头平仓 `is_long_exit`
  - 空头开仓 `is_short_entry`
  - 空头平仓 `is_short_exit`

提取订单参数：`tuple[float, int, int] (order_size, final_size_type, direction)`
- `order_size`：最终订单大小（正负表示方向）
- `final_size_type`：最终大小类型
- `direction`：交易方向限制

### 参数
- `position_now`：float，当前持仓数量（正数为多头，负数为空头，0为无持仓）
- `is_long_entry`：bool，是否有多头开仓信号
- `is_long_exit`：bool，是否有多头平仓信号
- `is_short_entry`：bool，是否有空头开仓信号
- `is_short_exit`：bool,是否有空头平仓信号
- `size`：float，信号对应的订单大小（非负数）
- `size_type`：int，订单大小类型，支持：Amount、Value、Percent
- `accumulate`：int，累积模式，参见 `AccumulationMode` 枚举：
  - `Disabled`：完全平仓或反转
  - `Both`：支持加仓和减仓
  - `AddOnly`：仅支持加仓
  - `RemoveOnly`：仅支持减仓
- `val_price_now`：float，当前估值价格，用于价值类型转换

### 逻辑
初始化返回值
- 订单大小：`order_size = 0`
- 交易方向：`direction = Direction.Both`
- 当前持仓绝对值：`abs_position_now = abs(position_now)`

根据持仓情况处理
- 当前持有多头头寸 `position_now > 0`
  - 有空头开仓信号 `is_short_entry == True`
    - 如果积累模式为双向/仅减少 `accumulate == AccumulationMode.Both/RemoveOnly`
      - `order_size = -size`（减去信号订单大小）
    - 否则（禁用累积/仅增加）
      - `order_size = -abs_position_now`（先平掉所有多头持仓）
      - 如果信号订单大小 `size` 不是 `NaN`（加上空头订单大小）
        - `size_type == SizeType.Percent`：`order_size -= size / val_price_now`
        - `size_type == SizeType.Value`：`order_size -= size`
      - `size_type = SizeType.Amount`
  - 否则如果有多头平仓信号 `is_long_exit == True`
    - `direction = Direction.LongOnly`（限制仓位只能在多头中）
    - 如果积累模式为双向/仅减少 `accumulate == AccumulationMode.Both/RemoveOnly`
      - `order_size = -size`（减去信号订单大小）
    - 否则（禁用累积/仅增加）
      - `order_size = -abs_position_now`（平掉所有多头持仓）
      - `size_type = SizeType.Amount`
  - 否则如果有多头开仓信号 `is_long_entry`（同方向）
    - `direction = Direction.LongOnly`（限制为只能多头持仓）
    - 如果积累模式为双向/仅增加 `accumulate == AccumulationMode.Both/AddOnly`
      - order_size = size（增加多头持仓）
- 当前持有空头头寸 `position_now < 0`
  - 有多头开仓信号 `is_long_entry == True`
    - 如果积累模式为双向/仅减少 `accumulate == AccumulationMode.Both/RemoveOnly`
      - `order_size = size`（减少空头持仓）
    - 否则（禁用累积/仅增加）
      - `order_size = abs_position_now`（先平掉所有空头持仓）
      - 如果信号订单大小 `size` 不是 `NaN`（加上多头订单大小）
        - `size_type == SizeType.Percent`：`order_size += size / val_price_now`
        - `size_type == SizeType.Value`：`order_size += size`
      - `size_type = SizeType.Amount`
  - 否则如果有空头平仓信号 `is_short_exit == True`
    - `direction = Direction.ShortOnly`（限制仓位只能在空头中）
    - 如果积累模式为双向/仅减少 `accumulate == AccumulationMode.Both/RemoveOnly`
      - `order_size = size`（减去信号订单大小）
    - 否则（禁用累积/仅增加）
      - `order_size = abs_position_now`（平掉所有空头持仓）
      - `size_type = SizeType.Amount`
  - 否则如果有空头开仓信号 `is_short_entry`（同方向）
    - `direction = Direction.ShortOnly`（限制为只能多头持仓）
    - 如果积累模式为双向/仅增加 `accumulate == AccumulationMode.Both/AddOnly`
      - order_size = size（增加空头持仓）
- 当前无持仓 `position_now == 0`
  - 有多头开仓信号 `is_long_entry == True`
    - `order_size = size`
  - 有空头开仓信号 `is_short_entry == True`
    - `order_size = -size`

### 源码
```python
@njit(cache=True)
def signals_to_size_nb(position_now: float,
                       is_long_entry: bool,
                       is_long_exit: bool,
                       is_short_entry: bool,
                       is_short_exit: bool,
                       size: float,
                       size_type: int,
                       accumulate: int,
                       val_price_now: float) -> tp.Tuple[float, int, int]:
    if size_type != SizeType.Amount and size_type != SizeType.Value and size_type != SizeType.Percent:
        raise ValueError("Only SizeType.Amount, SizeType.Value, and SizeType.Percent are supported")
    order_size = 0.
    direction = Direction.Both
    abs_position_now = abs(position_now)
    if is_less_nb(size, 0):
        raise ValueError("Negative size is not allowed. You must express direction using signals.")

    if position_now > 0:
        # We're in a long position
        if is_short_entry:
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.RemoveOnly:
                # Decrease the position
                order_size = -size
            else:
                # Reverse the position
                order_size = -abs_position_now
                if not np.isnan(size):
                    if size_type == SizeType.Percent:
                        raise ValueError(
                            "SizeType.Percent does not support position reversal using signals")
                    if size_type == SizeType.Value:
                        order_size -= size / val_price_now
                    else:
                        order_size -= size
                size_type = SizeType.Amount
        elif is_long_exit:
            direction = Direction.LongOnly
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.RemoveOnly:
                # Decrease the position
                order_size = -size
            else:
                # Close the position
                order_size = -abs_position_now
                size_type = SizeType.Amount
        elif is_long_entry:
            direction = Direction.LongOnly
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.AddOnly:
                # Increase the position
                order_size = size
    elif position_now < 0:
        # We're in a short position
        if is_long_entry:
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.RemoveOnly:
                # Decrease the position
                order_size = size
            else:
                # Reverse the position
                order_size = abs_position_now
                if not np.isnan(size):
                    if size_type == SizeType.Percent:
                        raise ValueError("SizeType.Percent does not support position reversal using signals")
                    if size_type == SizeType.Value:
                        order_size += size / val_price_now
                    else:
                        order_size += size
                size_type = SizeType.Amount
        elif is_short_exit:
            direction = Direction.ShortOnly
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.RemoveOnly:
                # Decrease the position
                order_size = size
            else:
                # Close the position
                order_size = abs_position_now
                size_type = SizeType.Amount
        elif is_short_entry:
            direction = Direction.ShortOnly
            if accumulate == AccumulationMode.Both or accumulate == AccumulationMode.AddOnly:
                # Increase the position
                order_size = -size
    else:
        if is_long_entry:
            # Open long position
            order_size = size
        elif is_short_entry:
            # Open short position
            order_size = -size

    return order_size, size_type, direction
```

## should_update_stop_nb
根据止损更新模式 `upon_stop_update` 判断是否应该更新止损价位。

参数
- `stop`：float，新的止损值，可能为 `NaN`
- `upon_stop_update`：int，止损更新模式，参见 `StopUpdateMode` 枚举
  - `Override`：有效值时覆盖
  - `OverrideNaN`：包括 `NaN` 值也覆盖
  - 其他：不更新
    
返回：bool，是否应该更新止损价位

### 源码
```python
@njit(cache=True)
def should_update_stop_nb(stop: float, upon_stop_update: int) -> bool:
    if upon_stop_update == StopUpdateMode.Override or upon_stop_update == StopUpdateMode.OverrideNaN:
        if not np.isnan(stop) or upon_stop_update == StopUpdateMode.OverrideNaN:
            return True
    return False
```

## get_stop_price_nb
根据
- 当前持仓方向 `position_now`
- 止损/止盈百分比 `stop`
- 价格范围 `open`，`low`，`high`
- 受否在价格下方触发 `hit_below`

计算实际的止损/止盈价格。

### 逻辑
- 止损/止盈百分比 `stop` 必须大于 0
- 如果：多头持仓向下止损 `position_now > 0 and hit_below` 或者空头持仓向上止损 `position_now < 0 and not hit_below`
  - 止损价 `stop_price = stop_price * (1 - stop)`
  - 如果 `stop_price <= open`（开盘前已经触发）：返回 `open`
  - 如果 `low <= stop_price <= high`（盘中触发）：返回 `stop_price`
  - 否则返回 `np.nan`
- 如果：空头持仓向下止盈 `position_now < 0 and hit_below` 或者多头持仓向上止盈 `position_now > 0 and not hit_below`
  - 止盈价 `stop_price = stop_price * (1 + stop)`
  - 如果 `stop_price <= open`（开盘前已经触发）：返回 `open`
  - 如果 `low <= stop_price <= high`（盘中触发）：返回 `stop_price`
  - 否则返回 `np.nan`
- 否则返回 `np.nan`

### 源码
```python
@njit(cache=True)
def get_stop_price_nb(position_now: float,
                      stop_price: float,
                      stop: float,
                      open: float,
                      low: float,
                      high: float,
                      hit_below: bool) -> float:
    if stop < 0:
        raise ValueError("Stop value must be 0 or greater")
    if (position_now > 0 and hit_below) or (position_now < 0 and not hit_below):
        stop_price = stop_price * (1 - stop)
        if open <= stop_price:
            return open
        if low <= stop_price <= high:
            return stop_price
        return np.nan
    if (position_now < 0 and hit_below) or (position_now > 0 and not hit_below):
        stop_price = stop_price * (1 + stop)
        if stop_price <= open:
            return open
        if low <= stop_price <= high:
            return stop_price
        return np.nan
    return np.nan
```

## no_signal_func_nb
产生无交易信号 `(False, False, False, False)`。
- `(is_long_entry, is_long_exit, is_short_entry, is_short_exit)`

### 源码
```python
@njit
def no_signal_func_nb(c: SignalContext, *args) -> tp.Tuple[bool, bool, bool, bool]:
    return False, False, False, False
```

## no_adjust_sl_func_nb
返回参数*止损调整上下文对象* `c` 中的 `(curr_stop, curr_trail)`。

参数
- `c`：`AdjustSLContext`，止损调整上下文对象，包含当前止损状态
  - `i`：int，当前行索引：当前时间步的索引位置，范围[0, target_shape[0])
  - `col`：int，当前列索引：当前资产列的索引，范围[0, target_shape[1])，且在[from_col, to_col)内
  - `position_now`：float，当前持仓数量：当前时间点的资产持仓数量（正数多头，负数空头）
  - `val_price_now`：float，当前估值价格：当前时间点的资产估值价格，用于止损计算
  - `init_i`：int，初始止损行索引：止损首次设置时的时间步索引，保持不变
  - `init_price`：float，初始止损价格：止损首次设置时的价格基准，保持不变
  - `curr_i`：int，当前止损行索引：最近一次止损更新的时间步索引，随价格更新而变化
  - `curr_price`：float，当前止损价格：最近一次更新的止损价格，在跟踪止损中随价格上涨而上移
  - `curr_stop`：float，当前止损值：当前有效的止损价格，可由调整函数修改
  - `curr_trail`：bool，当前跟踪标志：当前是否为跟踪止损模式，可由调整函数修改
- `*args`：可变参数，额外的用户定义参数（此函数中未使用）

### 源码
```python
@njit
def no_adjust_sl_func_nb(c: AdjustSLContext, *args) -> tp.Tuple[float, bool]:
    return c.curr_stop, c.curr_trail
```

## no_adjust_tp_func_nb
返回参数*止盈调整上下文对象* `c` 中的 `curr_stop`。

参数
- `c`：`AdjustTPContext`，止盈调整上下文对象，包含当前止盈状态
  - `i`：int，当前行索引：当前时间步的索引位置，范围[0, target_shape[0])
  - `col`：int，当前列索引：当前资产列的索引，范围[0, target_shape[1])，且在[from_col, to_col)内
  - `position_now`：float，当前持仓数量：当前时间点的资产持仓数量（正数多头，负数空头）
  - `val_price_now`：float，当前估值价格：当前时间点的资产估值价格，用于止盈计算
  - `init_i`：int，初始止盈行索引：止盈首次设置时的时间步索引，保持不变
  - `init_price`：float，初始止盈价格：止盈首次设置时的价格基准，保持不变
  - `curr_stop`：float，当前止盈值：当前有效的止盈价格，可由调整函数修改
- `*args`：可变参数，额外的用户定义参数（此函数中未使用）

### 源码
```python
@njit
def no_adjust_tp_func_nb(c: AdjustTPContext, *args) -> float:
    return c.curr_stop
```

## simulate_from_signal_func_nb

### 逻辑
- 验证
  - 分组包括了所有资产：分组序列 `group_lens` 各元素的和要与 `target_shape[1]` 相同
  - 是否真正具有非平凡的分组 `cash_sharing = np.any(group_lens > 1)`
  - 现金配置与分组的兼容性
    - `cash_sharing` 即现金在组内共享时，`init_cash` 和 `group_lens` 的长度须相同
    - `!cash_sharing` 即非现金共享时，`init_cash` 的长度必须是资产数 `target_shape[1]`
- 初始化
  - 订单和日志记录数组 `order_records`，`log_records`
  - `init_cash`
  - 投资组合状态数组 `last_position`，`last_debt`，`last_val_price`
  - 止损止盈相关状态 `sl_*`，`tp_*`
    - 如果启用止损止盈 `use_stops == True`：初始化长度为 `target_shape[1]` 的数组
    - 否则初始化为空数组

- 下面的循环嵌套结构为 *`分组——>时间步——>分组中的资产`*
- *循环处理每个分组 `for group in range(len(group_lens)):`*
  - 初始化组级现金状态 `cash_now = init_cash[group]`，`free_cash_now = init_cash[group]`
  - *循环处理每个时间步 `for i in range(target_shape[0]):`*
    - **1.解析组内每个资产的订单价格和估值价格**：
      - 对于 `price[i, col]`，根据下述情况缓存到 `price_arr[col]`：
        - +inf ——> close[i, col]
        - -inf 并且 open[i, col] 不为 inf ——> open[i, col]
        - -inf 并且 open[i, col] 为 inf 并且 i > 0 ——> close[i-1, col]
        - -inf 并且 open[i, col] 为 inf 并且 i= 0 ——> NaN
        - 否则，保持
      - 对于 `val_price[i, col]`，如果其不是 NaN 且 `!ffill_val_price`，根据下述情况缓存到 `last_val_price[col]`：
        - +inf ——> price_arr[col]
        - -inf 并且 i > 0 ——> close[i-1, col]
        - -inf 并且 i= 0 ——> NaN
        - 否则，保持
    - **2.更新组内每个订单的止损，生成最终订单参数，估算订单价值**。*循环处理每个资产 for k in range(group_len):*
      - `col = from_col + k`
      - 获取当前状态
        - 当前持仓：`position_now = last_position[col]`
        - 订单价格：`_price = price_arr[col]`
        - 滑点设置：`_slippage = slippage[i, col]`
        - 止损触发价格：`stop_price = np.nan`
      - **如果启用止损止盈 `use_stops == True`：计算止损/止盈价 `stop_price` 并更新 `sl_curr_i[col], sl_curr_price[col]`**
        - 构建止损调整上下文对象 `AdjustSLContext` 并返回其中 `(curr_stop, curr_trail)` 给 `sl_curr_stop[col], sl_curr_trail[col]`
        - 构建止盈调整上下文对象 `AdjustTPContext` 并返回其中 `curr_stop` 给 `tp_curr_stop[col]`
        - 如果 `sl_curr_stop[col]` 或 `tp_curr_stop[col]` 不为 `NaN`
          - 获取当前 OHLC 数据 `_open/high/low/close = open/high/low/close[i, col]`
          - 如果 `sl_curr_stop[col]` 不是 `NaN`
            - 使用 `get_stop_price_nb` 计算止损价 `stop_price`
          - 如果止损未触发 `stop_price == NaN` 并且 `tp_curr_stop[col]` 不是 `NaN`
            - 使用 `get_stop_price_nb` 计算止盈价 `stop_price` 
          - 如果 `sl_curr_stop[col]` 不是 `NaN` 并且 `sl_curr_trail[col] == True`，调整止损参考：
            - 如果当前多头持仓（`position_now > 0`）并且最高价大于参考止损参考价 `_high > sl_curr_price[col]`
              - `sl_curr_i[col] = i`
              - `sl_curr_price[col] = _high`
            - 如果当前空头持仓（`position_now < 0`）并且最低价小于参考止损参考价 `_low < sl_curr_price[col]`
              - `sl_curr_i[col] = i`
              - `sl_curr_price[col] = _low`
      - 获取累积模式 `_accumulate = accumulate[i, col]`
      - 如果启用止损止盈 `use_stops == True` 并且止损/止盈价 `stop_price` 非 `NaN`
        - 根据当前持仓 `position_now` 以及止损退出模式，使用函数 `generate_stop_signal_nb` 生成止损信号
        - 使用函数 `resolve_stop_price_and_slippage_nb` 确定最终的订单执行价格 `_price` 和滑点 `_slippage`。
      - 否则（未触发止损/止盈）
        - 构建信号生成上下文 `signal_ctx: SignalContent`
        - 调用用户自定义函数 `signal_func_nb` 生成交易信号
          - `is_long_entry, is_long_exit, is_short_entry, is_short_exit`
        - 如果有多头开仓 `is_long_entry` 或空头开仓信号 `is_short_entry`
          - **处理多头方向的开仓/平仓信号冲突**：同时多头开仓/平仓信号同时出现，使用 `resolve_signal_conflict_nb` 确定
            - `is_long_entry, is_long_exit`
          - **处理空头方向的开仓/平仓信号冲突**：同时空头开仓/平仓信号同时出现，使用 `resolve_signal_conflict_nb` 确定 
            - `is_short_entry, is_short_exit`
          - **处理多头/空头开仓信号同时出现的冲突**：同时出现多头/空头加仓信号时，使用 `resolve_dir_conflict_nb` 确定
            - `is_long_entry, is_short_entry`
          - **处理持仓与新开仓信号方向相反的情况**：使用 `resolve_opposite_entry_nb` 确定
            - `is_long_entry, is_long_exit, is_short_entry, is_short_exit, _accumulate`
      - **对于解决冲突的交易信号 `is_long_entry, is_long_exit, is_short_entry, is_short_exit` 使用 `signals_to_size_nb` 提取最终的订单参数**
        - `_size, _size_type, _direction`
      - 保存订单状态到临时数组 `price_arr, slippage_arr, size_arr, size_type_arr, direction_arr`
      - **如果现金共享 `cash_sharing`，估算订单价值到 `temp_order_value[k]`**：
        - 如果 `_size == 0`，则存 0
        - 否则
          - `_size_type == SizeType.Amount` ——> `_size * last_val_price[col]`
          - `_size_type == SizeType.Value` ——> `_size`
          - 否则（百分比类型）
            - `_size >= 0` ——> `_size * cash_now`（正向交易（买入/开仓），基于可用现金计算）
            - 否则（反向交易（卖出/平仓），基于持仓价值或最大风险敞口计算）
              - 计算持仓价值 `asset_value_now = last_position[col] * last_val_price[col]`
              - 如果 `_direction == Direction.LongOnly` ——> `_size * asset_value_now`
              - 否则，——> `_size * [2 * max(asset_value_now, 0) + max(free_cash_now, 0)]`
    - **3.订单排序以及组的总价值计算**
      - 如果现金共享 `cash_sharing` 
        - 如果 `auto_call_seq` 即按订单价值排序：对 `temp_order_value` 排序，将顺序存储到 `call_seq[i, from_col:to_col]`
        - 计算组的总价值 `value_now = cash_now + last_position[from_col] * last_val_price[from_col] + ...`；
    - **4.执行组内每个资产的订单**：*`for k in range(group_len)`*
      - `col = from_col + k`。如果 `cash_sharing` 即现金共享，`col = from_col + call_seq[i, col]`
      - 获取当前资产状态
        - 当前持仓：`position_now = last_position[col]`
        - 当前债务：`debt_now = last_debt[col]`
        - 当前估值价格：`val_price_now = last_val_price[col]`
      - 非现金共享模式为每个资产单独计算价值 `value_now`
        - `value_now = cash_now + position_now * val_price_now`
      - 使用 `order_nb` 创建订单对象 `order`
      - 根据当前组状态创建 `state: ProcessOrderState`
      - 使用 `process_order_nb` 执行订单 `order`
        - 返回 `order_result, new_state`
        - 会更新参数订单记录 `order_records` 和日志记录 `log_records`
      - 更新组级别状态 `state`，更新全局状态
      - 如果启用止损止盈 `use_stops == True` 且订单已经成功执行 `order_result.status == OrderStatus.Filled`
        - 如果当前无持仓 `position_now == 0`：清除所有止损/止盈设置 `sl_*/tp_*`
        - 否则（存在持仓）
          - 根据 `stop_entry_price[i, col]` 确定新的止损/止盈参考价 `new_init_price`
            - `StopEntryPrice.ValPrice` ——> `val_price_now`
            - `StopEntryPrice.Price` ——> `order.price`
            - `StopEntryPrice.FillPrice` ——> `order_result.price`
            - 否则 ——> `close[i, col]`
          - **根据持仓变化情况更新止损设置**
      - 更新资产级别状态 `last_position`，`last_debt`，`last_val_price`
- 返回订单记录 `order_records[:oidx]` 和日志记录 `log_records[:lidx]`


## dir_enex_signal_func_nb
```python
@njit
def dir_enex_signal_func_nb(c: SignalContext,
                            entries: tp.ArrayLike,
                            exits: tp.ArrayLike,
                            direction: tp.ArrayLike) -> tp.Tuple[bool, bool, bool, bool]:
    is_entry = flex_select_auto_nb(entries, c.i, c.col, c.flex_2d)
    is_exit = flex_select_auto_nb(exits, c.i, c.col, c.flex_2d)
    _direction = flex_select_auto_nb(direction, c.i, c.col, c.flex_2d)
    if _direction == Direction.LongOnly:
        return is_entry, is_exit, False, False
    if _direction == Direction.ShortOnly:
        return False, False, is_entry, is_exit
    return is_entry, False, is_exit, False
```

## ls_enex_signal_func_nb
```python
@njit
def ls_enex_signal_func_nb(c: SignalContext,
                           long_entries: tp.ArrayLike,
                           long_exits: tp.ArrayLike,
                           short_entries: tp.ArrayLike,
                           short_exits: tp.ArrayLike) -> tp.Tuple[bool, bool, bool, bool]:
    is_long_entry = flex_select_auto_nb(long_entries, c.i, c.col, c.flex_2d)
    is_long_exit = flex_select_auto_nb(long_exits, c.i, c.col, c.flex_2d)
    is_short_entry = flex_select_auto_nb(short_entries, c.i, c.col, c.flex_2d)
    is_short_exit = flex_select_auto_nb(short_exits, c.i, c.col, c.flex_2d)
    return is_long_entry, is_long_exit, is_short_entry, is_short_exit
```

## no_pre_func_nb
```python
@njit
def no_pre_func_nb(c: tp.NamedTuple, *args) -> tp.Args:
    return args
```

## no_order_func_nb
```python
@njit
def no_order_func_nb(c: OrderContext, *args) -> Order:
    return NoOrder
```

## no_post_func_nb
```python
@njit
def no_post_func_nb(c: tp.NamedTuple, *args) -> None:
    return None
```

## simulate_nb
列主序投资组合模拟。
- 组——>时间步——>订单

### 逻辑
构造模拟上下文对象 `pre_sim_ctx: SimulationContext`，并使用钩子函数 `pre_sim_func_nb` 转换为 `pre_sim_out`

***遍历每个组（所有列被划分为若干块）***
- 构建组上下文对象 `pre_group_ctx: GroupContent`，并使用钩子函数 `pre_group_func_nb` 转换为 `pre_group_out`
- ***遍历每个时间步***
  - 构建分段上下文对象 `pre_seg_ctx: SegmentContext`，并使用钩子函数 `pre_segment_func_nb` 转换为 `pre_segment_out`
  - 计算价值和收益率：
    - 如果现金共享则计算整个组，存到 `last_value/last_return[group]`；
    - 否则是计算每个列，存到 `last_value/last_return[col]`
  - 是否需要执行当前段中订单 `segment_mask[i, group]`
    - 按调用序列***遍历段中每个订单***
      - 构建订单上下文对象 `order_ctx: OrderContext`，并使用 `order_func_nb` 生成订单 `order`
      - 构建处理订单前的状态对象 `state: ProcessOrderState`
      - 使用 `process_order_nb` 处理订单并获得 `order_result, new_state`
        - 更新 `last_*[col/group]`
      - 构建订单后上下文对象 `post_order_ctx: PostOrderContext`，并使用钩子函数 `post_order_func_nb` 处理
  - 如果 `call_post_segment or segment_mask[i, group]`
    - 构建处理后分段上下文对象 `post_seg_ctx: SegmentContext`，并使用钩子函数 `post_segment_func_nb` 处理
- 构建处理后组上下文对象 `post_group_ctx: GroupContent`，并使用钩子函数 `post_group_func_nb` 处理
  
构造处理后模拟上下文对象 `post_sim_ctx: SimulationContext`，并使用钩子函数 `post_sim_func_nb` 处理

返回订单记录 `order_records` 和日志记录 `log_records`

## simulate_row_wise_nb
行主序投资组合模拟。
- 时间步——>组——>订单

### 逻辑
构造模拟上下文对象 `pre_sim_ctx: SimulationContext`，并使用钩子函数 `pre_sim_func_nb` 转换为 `pre_sim_out`

***遍历每个时间步***
- 构建行上下文对象 `pre_row_ctx: RowContext`，并使用钩子函数 `pre_row_func_nb` 转换为 `pre_row_out`
- ***遍历每个组***
  - 构建分段上下文对象 `pre_seg_ctx: SegmentContext`，并使用钩子函数 `pre_segment_func_nb` 转换为 `pre_segment_out`
  - 计算价值和收益率：
    - 如果现金共享则计算整个组，存到 `last_value/last_return[group]`；
    - 否则是计算每个列，存到 `last_value/last_return[col]`
  - 是否需要执行当前段中订单 `segment_mask[i, group]`
    - 按调用序列***遍历段中每个订单***
      - 构建订单上下文对象 `order_ctx: OrderContext`，并使用 `order_func_nb` 生成订单 `order`
      - 构建处理订单前的状态对象 `state: ProcessOrderState`
      - 使用 `process_order_nb` 处理订单并获得 `order_result, new_state`
        - 更新 `last_*[col/group]`
      - 构建订单后上下文对象 `post_order_ctx: PostOrderContext`，并使用钩子函数 `post_order_func_nb` 处理
  - 如果 `call_post_segment or segment_mask[i, group]`
    - 构建处理后分段上下文对象 `post_seg_ctx: SegmentContext`，并使用钩子函数 `post_segment_func_nb` 处理
- 构建处理后行上下文对象 `post_row_ctx: RowContext`，并使用钩子函数 `post_row_func_nb` 处理
  
构造处理后模拟上下文对象 `post_sim_ctx: SimulationContext`，并使用钩子函数 `post_sim_func_nb` 处理

返回订单记录 `order_records` 和日志记录 `log_records`

## no_flex_order_func_nb
```python
@njit
def no_flex_order_func_nb(c: FlexOrderContext, *args) -> tp.Tuple[int, Order]:
    return -1, NoOrder
```

## flex_simulate_nb
列主序投资组合模拟 `simulate_nb` 的灵活版本，区别在于可以使用 `lex_order_func_nb` 动态决定订单的执行顺序。
- 组——>时间步——>订单（顺序由 `lex_order_func_nb` 动态决定）

## flex_simulate_row_wise_nb
行主序投资组合模拟 `simulate_row_wise_nb` 的灵活版本，区别在于可以使用 `lex_order_func_nb` 动态决定订单的执行顺序。
- 组——>时间步——>订单（顺序由 `lex_order_func_nb` 动态决定）

# 交易记录处理

## get_trade_stats_nb
计算单笔交易的统计信息 
- *盈亏金额*：$size\left( {exit\_price - entry\_price} \right) - entry\_fees - exit\_fees$
- *收益率*：$\frac{盈亏金额}{{size \cdot entry\_price}}$
- 注意空头时，要 $size\left( {entry\_price - exit\_price} \right)$

参数
- `size`：float，交易数量（股数、手数等）
- `entry_price`：float，入场价格
- `entry_fees`：float，入场手续费
- `exit_price`：float，出场价格
- `exit_fees`：float，出场手续费
- `direction`：int，交易方向，使用 TradeDirection 枚举：
  - `Long`：多头交易
  - `Short`：空头交易

返回：`tuple[float, float]` (pnl, return) 
  - 盈亏金额

### 源码
```python
@njit(cache=True)
def get_trade_stats_nb(size: float,
                       entry_price: float,
                       entry_fees: float,
                       exit_price: float,
                       exit_fees: float,
                       direction: int) -> tp.Tuple[float, float]:
    entry_val = size * entry_price
    exit_val = size * exit_price
    val_diff = add_nb(exit_val, -entry_val)
    if val_diff != 0 and direction == TradeDirection.Short:
        val_diff *= -1
    pnl = val_diff - entry_fees - exit_fees
    ret = pnl / entry_val
    return pnl, ret
```

## fill_trade_record_nb
计算盈亏金额、收益率，并将其它信息一并存储到 `record`。

### 源码
```python
@njit(cache=True)
def fill_trade_record_nb(record: tp.Record,
                         id_: int,
                         col: int,
                         size: float,
                         entry_idx: int,
                         entry_price: float,
                         entry_fees: float,
                         exit_idx: int,
                         exit_price: float,
                         exit_fees: float,
                         direction: int,
                         status: int,
                         parent_id: int) -> None:

    # Calculate PnL and return
    pnl, ret = get_trade_stats_nb(
        size,
        entry_price,
        entry_fees,
        exit_price,
        exit_fees,
        direction
    )

    # Save trade
    record['id'] = id_
    record['col'] = col
    record['size'] = size
    record['entry_idx'] = entry_idx
    record['entry_price'] = entry_price
    record['entry_fees'] = entry_fees
    record['exit_idx'] = exit_idx
    record['exit_price'] = exit_price
    record['exit_fees'] = exit_fees
    record['pnl'] = pnl
    record['return'] = ret
    record['direction'] = direction
    record['status'] = status
    record['parent_id'] = parent_id
```

## fill_entry_trades_in_position_nb
获取订单记录 `order_records` 中组 `col` 列 `first_c~last_c` 中的入场订单，将它们转化为交易记录后存入 `trade_records`。
    
逻辑：
- 遍历持仓内的所有订单 `for c in range(first_c, last_c + 1)`
  - 获取订单记录
  - 忽略出场订单
  - 使用 `fill_trade_record_nb` 计算盈亏金额、收益率，并将其它信息一并存储到 `trade_records[tidx]`

### 源码
```python
@njit(cache=True)
def fill_entry_trades_in_position_nb(order_records: tp.RecordArray,
                                     col_map: tp.ColMap,
                                     col: int,
                                     first_c: int,
                                     last_c: int,
                                     first_entry_size: float,
                                     first_entry_fees: float,
                                     exit_idx: int,
                                     exit_size_sum: float,
                                     exit_gross_sum: float,
                                     exit_fees_sum: float,
                                     direction: int,
                                     status: int,
                                     parent_id: int,
                                     trade_records: tp.RecordArray,
                                     tidx: int) -> int:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens

    for c in range(first_c, last_c + 1):
        oidx = col_idxs[col_start_idxs[col] + c]
        record = order_records[oidx]
        order_side = record['side']

        if (direction == TradeDirection.Long and order_side == OrderSide.Sell) \
                or (direction == TradeDirection.Short and order_side == OrderSide.Buy):
            continue

        if c == first_c:
            entry_size = first_entry_size
            entry_fees = first_entry_fees
        else:
            entry_size = record['size']
            entry_fees = record['fees']

        exit_price = exit_gross_sum / exit_size_sum

        size_fraction = entry_size / exit_size_sum
        exit_fees = size_fraction * exit_fees_sum

        fill_trade_record_nb(
            trade_records[tidx],
            tidx,
            col,
            entry_size,
            record['idx'],
            record['price'],
            entry_fees,
            exit_idx,
            exit_price,
            exit_fees,
            direction,
            status,
            parent_id
        )
        tidx += 1

    return tidx
```

## get_entry_trades_nb
逻辑
- 遍历所有资产
  - 遍历当前资产的所有订单
    - 从 `order_records` 获取当前订单记录
    - 如果是该资产的第一个订单
      - 根据该订单类型 `OrderSide.Buy/Sell` 确定该组的方向 `direction` 为 `TradeDirection.Long/Short`
    - 根据当前方向 `direction` 和订单类型
      - 如果是持仓增加（多头买入或空头卖出）
        - `entry_size_sum += order_size, entry_gross_sum += order_size * order_price, entry_fees_sum += order_fees`
      - 如果是持仓减少（多头卖出或空头买入）
        - 如果完全平仓 `exit_size_sum + order_size == entry_size_sum`（出场数量等于入场数量）
          - 记录当前的订单位置为 `last_c`
          - 使用 `fill_entry_trades_in_position_nb` 获取订单记录 `order_records` 中资产 `col` 列 `first_c~last_c` 中的入场订单，将它们转化为交易记录后存入 `records`
        - 否则如果是部分平仓（出场数量小于入场数量）
          - `exit_size_sum += order_size, exit_gross_sum += order_size * order_price, exit_fees_sum += order_fees`
        - 否则（超额平仓）
          - 记录当前的订单位置为 `last_c`
          - 使用 `fill_entry_trades_in_position_nb` 获取订单记录 `order_records` 中资产 `col` 列 `first_c~last_c` 中的入场订单，将它们转化为交易记录后存入 `records`
          - 开新持仓：记录当前订单位置为 `first_c`，重置 `entry_size_sum, entry_gross_sum, entry_fees_sum` 等
  - 如果当前资产的所有订单结束后还未平仓
    - 使用 `fill_entry_trades_in_position_nb` 获取订单记录并将它们转化为交易记录后存入 `records`
- 返回 `records[:tidx]`

### 源码
```python
@njit(cache=True)
def get_entry_trades_nb(order_records: tp.RecordArray, close: tp.Array2d, col_map: tp.ColMap) -> tp.RecordArray:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    records = np.empty(len(order_records), dtype=trade_dt)
    tidx = 0
    parent_id = -1

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            continue
        last_id = -1
        in_position = False

        for c in range(col_len):
            oidx = col_idxs[col_start_idxs[col] + c]
            record = order_records[oidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            order_idx = record['idx']
            order_size = record['size']
            order_price = record['price']
            order_fees = record['fees']
            order_side = record['side']

            if order_size <= 0.:
                raise ValueError(size_zero_neg_err)
            if order_price <= 0.:
                raise ValueError(price_zero_neg_err)

            if not in_position:
                # New position opened
                first_c = c
                in_position = True
                parent_id += 1
                if order_side == OrderSide.Buy:
                    direction = TradeDirection.Long
                else:
                    direction = TradeDirection.Short
                entry_size_sum = 0.
                entry_gross_sum = 0.
                entry_fees_sum = 0.
                exit_size_sum = 0.
                exit_gross_sum = 0.
                exit_fees_sum = 0.
                first_entry_size = order_size
                first_entry_fees = order_fees

            if (direction == TradeDirection.Long and order_side == OrderSide.Buy) \
                    or (direction == TradeDirection.Short and order_side == OrderSide.Sell):
                # Position increased
                entry_size_sum += order_size
                entry_gross_sum += order_size * order_price
                entry_fees_sum += order_fees

            elif (direction == TradeDirection.Long and order_side == OrderSide.Sell) \
                    or (direction == TradeDirection.Short and order_side == OrderSide.Buy):
                if is_close_nb(exit_size_sum + order_size, entry_size_sum):
                    # Position closed
                    last_c = c
                    in_position = False
                    exit_size_sum = entry_size_sum
                    exit_gross_sum += order_size * order_price
                    exit_fees_sum += order_fees

                    # Fill trade records
                    tidx = fill_entry_trades_in_position_nb(
                        order_records,
                        col_map,
                        col,
                        first_c,
                        last_c,
                        first_entry_size,
                        first_entry_fees,
                        order_idx,
                        exit_size_sum,
                        exit_gross_sum,
                        exit_fees_sum,
                        direction,
                        TradeStatus.Closed,
                        parent_id,
                        records,
                        tidx
                    )
                elif is_less_nb(exit_size_sum + order_size, entry_size_sum):
                    # Position decreased
                    exit_size_sum += order_size
                    exit_gross_sum += order_size * order_price
                    exit_fees_sum += order_fees
                else:
                    # Position closed
                    last_c = c
                    remaining_size = add_nb(entry_size_sum, -exit_size_sum)
                    exit_size_sum = entry_size_sum
                    exit_gross_sum += remaining_size * order_price
                    exit_fees_sum += remaining_size / order_size * order_fees

                    # Fill trade records
                    tidx = fill_entry_trades_in_position_nb(
                        order_records,
                        col_map,
                        col,
                        first_c,
                        last_c,
                        first_entry_size,
                        first_entry_fees,
                        order_idx,
                        exit_size_sum,
                        exit_gross_sum,
                        exit_fees_sum,
                        direction,
                        TradeStatus.Closed,
                        parent_id,
                        records,
                        tidx
                    )

                    # New position opened
                    first_c = c
                    parent_id += 1
                    if order_side == OrderSide.Buy:
                        direction = TradeDirection.Long
                    else:
                        direction = TradeDirection.Short
                    entry_size_sum = add_nb(order_size, -remaining_size)
                    entry_gross_sum = entry_size_sum * order_price
                    entry_fees_sum = entry_size_sum / order_size * order_fees
                    first_entry_size = entry_size_sum
                    first_entry_fees = entry_fees_sum
                    exit_size_sum = 0.
                    exit_gross_sum = 0.
                    exit_fees_sum = 0.

        if in_position and is_less_nb(exit_size_sum, entry_size_sum):
            # Position hasn't been closed
            last_c = col_len - 1
            remaining_size = add_nb(entry_size_sum, -exit_size_sum)
            exit_size_sum = entry_size_sum
            exit_gross_sum += remaining_size * close[close.shape[0] - 1, col]

            # Fill trade records
            tidx = fill_entry_trades_in_position_nb(
                order_records,
                col_map,
                col,
                first_c,
                last_c,
                first_entry_size,
                first_entry_fees,
                close.shape[0] - 1,
                exit_size_sum,
                exit_gross_sum,
                exit_fees_sum,
                direction,
                TradeStatus.Open,
                parent_id,
                records,
                tidx
            )

    return records[:tidx]
```

## get_exit_trades_nb
逻辑
- 遍历所有资产
  - 遍历当前资产的所有订单
    - 从 `order_records` 获取当前订单记录
    - 如果是该资产的第一个订单
      - 根据该订单类型 `OrderSide.Buy/Sell` 确定该组的方向 `direction` 为 `TradeDirection.Long/Short`
    - 根据当前方向 `direction` 和订单类型
      - 如果是持仓增加（多头买入或空头卖出）
        - `entry_size_sum += order_size, entry_gross_sum += order_size * order_price, entry_fees_sum += order_fees`
      - 如果是持仓减少（多头卖出或空头买入）
        - 如果订单数量不大于入场数量 `order_size <= entry_size_sum`（订单数量不大于入场数量）
          - 使用 `fill_trade_record_nb` 计算盈亏金额、收益率，并将其它信息一并存储到 `records[tidx]`
          - 如果完全平仓 `order_size == entry_size_sum`
            - 重置持仓状态
          - 否则（部分平仓）
            - 按比例 $\frac{{entry\_size\_sum - order\_size}}{entry\_size\_sum}$ 缩减入场情况
        - 否则（超额平仓，先平仓再开立反向新仓）
          - 使用 `fill_trade_record_nb` 计算盈亏金额、收益率，并将其它信息一并存储到 `records[tidx]`
          - 开新持仓：记录当前订单位置为 `first_c`，重置 `entry_size_sum, entry_gross_sum, entry_fees_sum` 等
  - 如果当前资产的所有订单结束后还未平仓
    - 使用 `fill_trade_record_nb` 计算盈亏金额、收益率，并将其它信息一并存储到 `records[tidx]`
- 返回 `records[:tidx]`

### 源码
```python
@njit(cache=True)
def get_exit_trades_nb(order_records: tp.RecordArray, close: tp.Array2d, col_map: tp.ColMap) -> tp.RecordArray:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    records = np.empty(len(order_records), dtype=trade_dt)
    tidx = 0
    parent_id = -1

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            continue
        last_id = -1
        in_position = False

        for c in range(col_len):
            oidx = col_idxs[col_start_idxs[col] + c]
            record = order_records[oidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            i = record['idx']
            order_size = record['size']
            order_price = record['price']
            order_fees = record['fees']
            order_side = record['side']

            if order_size <= 0.:
                raise ValueError(size_zero_neg_err)
            if order_price <= 0.:
                raise ValueError(price_zero_neg_err)

            if not in_position:
                # Trade opened
                in_position = True
                entry_idx = i
                if order_side == OrderSide.Buy:
                    direction = TradeDirection.Long
                else:
                    direction = TradeDirection.Short
                parent_id += 1
                entry_size_sum = 0.
                entry_gross_sum = 0.
                entry_fees_sum = 0.

            if (direction == TradeDirection.Long and order_side == OrderSide.Buy) \
                    or (direction == TradeDirection.Short and order_side == OrderSide.Sell):
                # Position increased
                entry_size_sum += order_size
                entry_gross_sum += order_size * order_price
                entry_fees_sum += order_fees

            elif (direction == TradeDirection.Long and order_side == OrderSide.Sell) \
                    or (direction == TradeDirection.Short and order_side == OrderSide.Buy):
                if is_close_or_less_nb(order_size, entry_size_sum):
                    # Trade closed
                    if is_close_nb(order_size, entry_size_sum):
                        exit_size = entry_size_sum
                    else:
                        exit_size = order_size
                    exit_price = order_price
                    exit_fees = order_fees
                    exit_idx = i

                    # Take a size-weighted average of entry price
                    entry_price = entry_gross_sum / entry_size_sum

                    # Take a fraction of entry fees
                    size_fraction = exit_size / entry_size_sum
                    entry_fees = size_fraction * entry_fees_sum

                    fill_trade_record_nb(
                        records[tidx],
                        tidx,
                        col,
                        exit_size,
                        entry_idx,
                        entry_price,
                        entry_fees,
                        exit_idx,
                        exit_price,
                        exit_fees,
                        direction,
                        TradeStatus.Closed,
                        parent_id
                    )
                    tidx += 1

                    if is_close_nb(order_size, entry_size_sum):
                        # Position closed
                        entry_idx = -1
                        direction = -1
                        in_position = False
                    else:
                        # Position decreased, previous orders have now less impact
                        size_fraction = (entry_size_sum - order_size) / entry_size_sum
                        entry_size_sum *= size_fraction
                        entry_gross_sum *= size_fraction
                        entry_fees_sum *= size_fraction
                else:
                    # Trade reversed
                    # Close current trade
                    cl_exit_size = entry_size_sum
                    cl_exit_price = order_price
                    cl_exit_fees = cl_exit_size / order_size * order_fees
                    cl_exit_idx = i

                    # Take a size-weighted average of entry price
                    entry_price = entry_gross_sum / entry_size_sum

                    # Take a fraction of entry fees
                    size_fraction = cl_exit_size / entry_size_sum
                    entry_fees = size_fraction * entry_fees_sum

                    fill_trade_record_nb(
                        records[tidx],
                        tidx,
                        col,
                        cl_exit_size,
                        entry_idx,
                        entry_price,
                        entry_fees,
                        cl_exit_idx,
                        cl_exit_price,
                        cl_exit_fees,
                        direction,
                        TradeStatus.Closed,
                        parent_id
                    )
                    tidx += 1

                    # Open a new trade
                    entry_size_sum = order_size - cl_exit_size
                    entry_gross_sum = entry_size_sum * order_price
                    entry_fees_sum = order_fees - cl_exit_fees
                    entry_idx = i
                    if direction == TradeDirection.Long:
                        direction = TradeDirection.Short
                    else:
                        direction = TradeDirection.Long
                    parent_id += 1

        if in_position and is_less_nb(-entry_size_sum, 0):
            # Trade hasn't been closed
            exit_size = entry_size_sum
            exit_price = close[close.shape[0] - 1, col]
            exit_fees = 0.
            exit_idx = close.shape[0] - 1

            # Take a size-weighted average of entry price
            entry_price = entry_gross_sum / entry_size_sum

            # Take a fraction of entry fees
            size_fraction = exit_size / entry_size_sum
            entry_fees = size_fraction * entry_fees_sum

            fill_trade_record_nb(
                records[tidx],
                tidx,
                col,
                exit_size,
                entry_idx,
                entry_price,
                entry_fees,
                exit_idx,
                exit_price,
                exit_fees,
                direction,
                TradeStatus.Open,
                parent_id
            )
            tidx += 1

    return records[:tidx]
```

## trade_winning_streak_nb
计算 `records` 中每笔交易的当前连胜次数。

参数：
- `records`：RecordArray，交易记录数组，必须包含 `'pnl'` 字段（盈亏信息）
    
返回：Array1d[int64]，每笔交易对应的连胜次数数组
- 如果当前交易盈利：连胜次数 = 之前连胜次数 + 1
- 如果当前交易亏损：连胜次数重置为 0

```python
@njit(cache=True)
def trade_winning_streak_nb(records: tp.RecordArray) -> tp.Array1d:
    out = np.full(len(records), 0, dtype=np.int64)
    curr_rank = 0
    for i in range(len(records)):
        if records[i]['pnl'] > 0:
            curr_rank += 1
        else:
            curr_rank = 0
        out[i] = curr_rank
    return out
```

## trade_losing_streak_nb
计算 `records` 中每笔交易的当前连败次数。

参数：
- `records`：RecordArray，交易记录数组，必须包含 `'pnl'` 字段（盈亏信息）
    
返回：Array1d[int64]，每笔交易对应的连败次数数组
- 如果当前交易亏损：连败次数 = 之前连败次数 + 1
- 如果当前交易盈利：连败次数重置为 0

```python
@njit(cache=True)
def trade_losing_streak_nb(records: tp.RecordArray) -> tp.Array1d:
    out = np.full(len(records), 0, dtype=np.int64)
    curr_rank = 0
    for i in range(len(records)):
        if records[i]['pnl'] < 0:
            curr_rank += 1
        else:
            curr_rank = 0
        out[i] = curr_rank
    return out
```

# 仓位记录

## fill_position_record_nb
聚合 `trade_records` 中的交易信息为一个统一的交易，然后计算盈亏金额、收益率，并将其它信息一并存储到 `record`。

### 源码
```python
@njit(cache=True)
def fill_position_record_nb(record: tp.Record, id_: int, trade_records: tp.RecordArray) -> None:
    """Fill a position record by aggregating trade records."""
    # Aggregate trades
    col = trade_records['col'][0]
    size = np.sum(trade_records['size'])
    entry_idx = trade_records['entry_idx'][0]
    entry_price = np.sum(trade_records['size'] * trade_records['entry_price']) / size
    entry_fees = np.sum(trade_records['entry_fees'])
    exit_idx = trade_records['exit_idx'][-1]
    exit_price = np.sum(trade_records['size'] * trade_records['exit_price']) / size
    exit_fees = np.sum(trade_records['exit_fees'])
    direction = trade_records['direction'][-1]
    status = trade_records['status'][-1]
    pnl, ret = get_trade_stats_nb(
        size,
        entry_price,
        entry_fees,
        exit_price,
        exit_fees,
        direction
    )

    # Save position
    record['id'] = id_
    record['col'] = col
    record['size'] = size
    record['entry_idx'] = entry_idx
    record['entry_price'] = entry_price
    record['entry_fees'] = entry_fees
    record['exit_idx'] = exit_idx
    record['exit_price'] = exit_price
    record['exit_fees'] = exit_fees
    record['pnl'] = pnl
    record['return'] = ret
    record['direction'] = direction
    record['status'] = status
    record['parent_id'] = id_
```

## copy_trade_record_nb
```python
@njit(cache=True)
def copy_trade_record_nb(record: tp.Record, trade_record: tp.Record) -> None:
    record['id'] = trade_record['id']
    record['col'] = trade_record['col']
    record['size'] = trade_record['size']
    record['entry_idx'] = trade_record['entry_idx']
    record['entry_price'] = trade_record['entry_price']
    record['entry_fees'] = trade_record['entry_fees']
    record['exit_idx'] = trade_record['exit_idx']
    record['exit_price'] = trade_record['exit_price']
    record['exit_fees'] = trade_record['exit_fees']
    record['pnl'] = trade_record['pnl']
    record['return'] = trade_record['return']
    record['direction'] = trade_record['direction']
    record['status'] = trade_record['status']
    record['parent_id'] = trade_record['parent_id']
```

## get_positions_nb
通过聚合交易记录生成持仓记录。

算法原理：
- 按列分组处理交易记录，确保每个资产独立分析
- 通过parent_id字段识别属于同一持仓的交易
- 聚合同一持仓的所有交易，计算综合的持仓指标
- 支持分批建仓和分批平仓的复杂交易模式

参数说明：
- `trade_records`：tp.RecordArray，交易记录数组，包含完整的交易信息，数据类型为trade_dt
- `col_map`：tp.ColMap，列映射结构，定义交易记录按列的分组信息
  - 格式：(col_idxs, col_lens)元组
  - col_idxs: 按列分组的索引数组
  - col_lens: 每列的记录数量数组
    
返回：tp.RecordArray，持仓记录数组，数据类型为trade_dt，包含聚合后的持仓信息

逻辑
- 遍历每个资产
  - 遍历当前资产的所有交易
    - 对于父id `parent_id` 相同的交易，使用 `fill_position_record_nb` 聚合交易信息为一个统一的交易，然后计算盈亏金额、收益率，并将其它信息一并存储到 `records[pidx]`
- 返回 `records`

### 源码
```python
@njit(cache=True)
def get_positions_nb(trade_records: tp.RecordArray, col_map: tp.ColMap) -> tp.RecordArray:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    records = np.empty(len(trade_records), dtype=trade_dt)
    pidx = 0
    from_tidx = -1

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            continue
        last_id = -1
        last_position_id = -1

        for c in range(col_len):
            tidx = col_idxs[col_start_idxs[col] + c]
            record = trade_records[tidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            parent_id = record['parent_id']

            if parent_id != last_position_id:
                if last_position_id != -1:
                    if tidx - from_tidx > 1:
                        fill_position_record_nb(records[pidx], pidx, trade_records[from_tidx:tidx])
                    else:
                        # Speed up
                        copy_trade_record_nb(records[pidx], trade_records[from_tidx])
                        records[pidx]['id'] = pidx
                        records[pidx]['parent_id'] = pidx
                    pidx += 1
                from_tidx = tidx
                last_position_id = parent_id

        if tidx - from_tidx > 0:
            fill_position_record_nb(records[pidx], pidx, trade_records[from_tidx:tidx + 1])
        else:
            # Speed up
            copy_trade_record_nb(records[pidx], trade_records[from_tidx])
            records[pidx]['id'] = pidx
            records[pidx]['parent_id'] = pidx
        pidx += 1

    return records[:pidx]
```

# 资产持仓

## get_long_size_nb
计算多头持仓净变化量
- 双方均为空头/空仓（position_before ≤ 0, position_now ≤ 0）：返回0，因为不涉及多头变化
- 多头转空头（position_before ≥ 0, position_now < 0）：返回 -position_before，表示减少了所有多头持仓
- 空头转多头（position_before < 0, position_now ≥ 0）：返回 position_now，表示新建多头持仓
- 双方均为多头（其他情况）：返回净变化量 = position_now - position_before

```python
@njit(cache=True)
def get_long_size_nb(position_before: float, position_now: float) -> float:
    if position_before <= 0 and position_now <= 0:
        return 0.
    if position_before >= 0 and position_now < 0:
        return -position_before
    if position_before < 0 and position_now >= 0:
        return position_now
    return add_nb(position_now, -position_before)
```

## get_short_size_nb
计算空头持仓净变化量
- 双方均为多头/空仓（position_before ≥ 0, position_now ≥ 0）：返回0，因为不涉及空头变化
- 多头转空头（position_before ≥ 0, position_now < 0）：返回 -position_now，表示新建空头持仓（负值）
- 空头转多头（position_before < 0, position_now ≥ 0）：返回 position_before，表示减少空头持仓（正值）
- 双方均为空头（其他情况）：返回净变化量 = position_before - position_now

```python
def get_short_size_nb(position_before: float, position_now: float) -> float:
    if position_before >= 0 and position_now >= 0:
        return 0.
    if position_before >= 0 and position_now < 0:
        return -position_now
    if position_before < 0 and position_now >= 0:
        return position_before
    return add_nb(position_before, -position_now)
```

## asset_flow_nb
计算每列的资产流量序列

逻辑
- 遍历所有组
  - 遍历每组的所有订单
    - 获取订单方向 `side` 和订单大小 `size`
    - 计算累积持仓 `new_position_now = position_now - size`
    - 根据参数 `direction == LongOnly/ShortOnly` 使用 `get_long/short_size_nb` 对 `(position_now, new_position_now)` 计算持仓变化 `asset_flow`
    - 累加到输出矩阵 `out` 的对应项
    - 更新持仓 `position_now = new_position_now`
- 返回流量矩阵 `out`

参数
- `target_shape`：tp.Shape，目标形状元组 (时间步数, 资产数量)
- `order_records`：tp.RecordArray，订单记录数组，包含完整的交易执行信息
- `col_map`：tp.ColMap，列映射结构，定义订单记录按列的分组信息
  - 格式：(col_idxs, col_lens)元组
  - `col_idxs`：按列分组的索引数组
  - `col_lens`：每列的记录数量数组
- `direction`：int，资产流量计算方向，使用Direction枚举值
  - `Direction.LongOnly (0)`：仅计算多头流量
  - `Direction.ShortOnly (1)`：仅计算空头流量  
  - `Direction.Both (2)`：计算双向流量（默认）
    
返回：tp.Array2d，资产流量矩阵，形状为 `target_shape`

### 源码
```python
@njit(cache=True)
def asset_flow_nb(target_shape: tp.Shape,
                  order_records: tp.RecordArray,
                  col_map: tp.ColMap,
                  direction: int) -> tp.Array2d:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    out = np.full(target_shape, 0., dtype=np.float64)

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            continue
        last_id = -1
        position_now = 0.

        for c in range(col_len):
            oidx = col_idxs[col_start_idxs[col] + c]
            record = order_records[oidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            i = record['idx']
            side = record['side']
            size = record['size']

            if side == OrderSide.Sell:
                size *= -1
            new_position_now = add_nb(position_now, size)
            if direction == Direction.LongOnly:
                asset_flow = get_long_size_nb(position_now, new_position_now)
            elif direction == Direction.ShortOnly:
                asset_flow = get_short_size_nb(position_now, new_position_now)
            else:
                asset_flow = size
            out[i, col] = add_nb(out[i, col], asset_flow)
            position_now = new_position_now
    return out
```

## assets_nb
根据资产流量 `asset_flow` 计算每列的持仓序列。

逻辑
- 遍历 `asset_flow` 的每一列
  - 初始化 `position_now = 0`
  - 遍历 `asset_flow` 的每一行
    - `position_now = position_now + asset_flow[i, col]`
    - `out[i, col] = position_now`
- 返回 `out`

参数
- `asset_flow`：tp.Array2d，资产流量矩阵，形状为 (时间步数, 资产数量)
  - 通常由 `asset_flow_nb` 函数生成
  - 正值：资产流入（建仓或增仓）
  - 负值：资产流出（减仓或平仓）
  - 零值：无资产流动
    
返回：tp.Array2d，持仓矩阵，形状与 `asset_flow` 相同

```python
@njit(cache=True)
def assets_nb(asset_flow: tp.Array2d) -> tp.Array2d:
    out = np.empty_like(asset_flow)
    for col in range(asset_flow.shape[1]):
        position_now = 0.
        for i in range(asset_flow.shape[0]):
            flow_value = asset_flow[i, col]
            position_now = add_nb(position_now, flow_value)
            out[i, col] = position_now
    return out
```

## i_group_any_reduce_nb
```python
@njit(cache=True)
def i_group_any_reduce_nb(i: int, group: int, a: tp.Array1d) -> bool:
    return np.any(a)
```

## position_mask_grouped_nb
指定分组 `group_lens` 对 `position_mask` 进行归约：组中存在非零即为 `True`。
```python
@njit
def position_mask_grouped_nb(position_mask: tp.Array2d, group_lens: tp.Array1d) -> tp.Array2d:
    return generic_nb.squeeze_grouped_nb(position_mask, group_lens, i_group_any_reduce_nb).astype(np.bool_)
```

In [ ]:
import numpy as np
import pandas as pd
from vectorbt.portfolio.nb import position_mask_grouped_nb

# 创建示例持仓掩码矩阵（3个时间点，5个资产）
position_mask = np.array([
    [True,  False, True,  False, True ],  # 时间0：资产0,2,4有持仓
    [False, False, False, False, False],  # 时间1：所有资产都无持仓
    [True,  True,  False, True,  False], # 时间2：资产0,1,3有持仓
])

# 定义分组：第1组3个资产，第2组2个资产
group_lens = np.array([3, 2])

# 计算分组后的持仓掩码
grouped_mask = position_mask_grouped_nb(position_mask, group_lens)

# 转换为DataFrame分析
df_original = pd.DataFrame(position_mask, 
                            columns=['A1', 'A2', 'A3', 'B1', 'B2'],
                            index=['时间0', '时间1', '时间2'])
df_grouped = pd.DataFrame(grouped_mask, 
                            columns=['组合A', '组合B'],
                            index=['时间0', '时间1', '时间2'])

print("原始持仓掩码（资产级别）:")
print(df_original)
print("\n分组持仓掩码（组合级别）:")
print(df_grouped)

## group_mean_reduce_nb
```python
@njit(cache=True)
def group_mean_reduce_nb(group: int, a: tp.Array1d) -> float:
    return np.mean(a)
```

## position_coverage_grouped_nb
指定分组 `group_lens` 对 `position_mask` 进行归约：组中取平均。


```python
@njit
def position_coverage_grouped_nb(position_mask: tp.Array2d, group_lens: tp.Array1d) -> tp.Array2d:
    return generic_nb.reduce_grouped_nb(position_mask, group_lens, group_mean_reduce_nb)
```

# 资金管理

## get_free_cash_diff_nb
根据持仓变化 `position_before` ——> `position_now` 计算债务更新和自由现金变化。

### 参数和返回
参数
- `position_before`：float，交易前的持仓数量
  - 正数：多头持仓
  - 负数：空头持仓
  - 零：空仓状态
- `position_now`：float，交易后的持仓数量
  - 正数：多头持仓
  - 负数：空头持仓
  - 零：空仓状态
- `debt_now`：float，当前的债务余额
  - 正值：存在债务（通常来自空头交易）
  - 零值：无债务
  - 用于计算空头交易的保证金要求
- `price`：float，交易价格
- `fees`：float， 交易手续费
    
返回：tp.Tuple[float, float]
- `new_debt`：更新后的债务余额
- `free_cash_diff`：自由现金流变化量

### 逻辑
- 计算持仓变化 `size = position_now - position_before`
- 计算持仓变化产生的基础现金 `final_cash = -size * price - fees`（不考虑空头时的保证金）
- 根据持仓变化 `size`
  - 如果 `size == 0`（持仓无变化）
    - `new_debt = debt_now, free_cash_diff = 0`
  - 否则如果 `size > 0`
    - 如果 `position_before < 0`（之前为空头）
      - 如果 `position_now < 0`（空头减仓）
        - `short_size = size`
      - 否则（空头转多头）
        - `short_size = - position_before`
      - 空头持仓的平均入场价格：`avg_entry_price = debt_now / abs(position_before)`
      - 空头金额减少量：`debt_diff = short_size * avg_entry_price`
      - 更新空头金额：`new_debt = debt_now - debt_diff`
      - 持仓变化产生的现金：`free_cash_diff = 2 * debt_diff + final_cash`
    - 否则（之前为多头）
      - `new_debt = debt_now, free_cash_diff = final_cash`
  - 否则（`size < 0`）
    - 如果 `position_now < 0`
      - 如果 `position_before < 0`（空头增仓）
        - `short_size = - size`
      - 否则（多头转空头）
        - `short_size = - position_now`
      - 新增空头需要的保证金：`short_value = short_size * price`
      - 更新空头金额：`new_debt = debt_now + short_value`
      - 持仓变化产生的现金：`free_cash_diff = final_cash - 2 * short_value`
    - 否则 `new_debt = debt_now, free_cash_diff = final_cash`
- 返回 `new_debt, free_cash_diff`

### 源码
```python
@njit(cache=True)
def get_free_cash_diff_nb(position_before: float,
                          position_now: float,
                          debt_now: float,
                          price: float,
                          fees: float) -> tp.Tuple[float, float]:
    size = add_nb(position_now, -position_before)
    final_cash = -size * price - fees
    if is_close_nb(size, 0):
        new_debt = debt_now
        free_cash_diff = 0.
    elif size > 0:
        if position_before < 0:
            if position_now < 0:
                short_size = abs(size)
            else:
                short_size = abs(position_before)
            avg_entry_price = debt_now / abs(position_before)
            debt_diff = short_size * avg_entry_price
            new_debt = add_nb(debt_now, -debt_diff)
            free_cash_diff = add_nb(2 * debt_diff, final_cash)
        else:
            new_debt = debt_now
            free_cash_diff = final_cash
    else:
        if position_now < 0:
            if position_before < 0:
                short_size = abs(size)
            else:
                short_size = abs(position_now)
            short_value = short_size * price
            new_debt = debt_now + short_value
            free_cash_diff = add_nb(final_cash, -2 * short_value)
        else:
            new_debt = debt_now
            free_cash_diff = final_cash
    return new_debt, free_cash_diff
```

## cash_flow_nb
基于 `get_free_cash_diff_nb` 计算每个资产的现金流序列（每个资产有多个订单）。

逻辑：
- 遍历每个资产
  - 遍历所有订单
    - 提取订单信息
    - 计算新持仓 `new_position_now = position_now + size`
    - 使用 `get_free_cash_diff_nb` 来根据持仓变化 `position_now` ——> `new_position_now` 计算债务更新 `debt_now` 和自由现金变化 `cash_flow`
    - 将 `cash_flow` 累加到 `out` 对应位置
    - `position_now = new_position_now`
- 返回 `out`

### 源码
```python
@njit(cache=True)
def cash_flow_nb(target_shape: tp.Shape,
                 order_records: tp.RecordArray,
                 col_map: tp.ColMap,
                 free: bool) -> tp.Array2d:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    out = np.full(target_shape, 0., dtype=np.float64)

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            continue
        last_id = -1
        position_now = 0.
        debt_now = 0.

        for c in range(col_len):
            oidx = col_idxs[col_start_idxs[col] + c]
            record = order_records[oidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            i = record['idx']
            side = record['side']
            size = record['size']
            price = record['price']
            fees = record['fees']

            if side == OrderSide.Sell:
                size *= -1
            new_position_now = add_nb(position_now, size)
            if free:
                debt_now, cash_flow = get_free_cash_diff_nb(
                    position_now,
                    new_position_now,
                    debt_now,
                    price,
                    fees
                )
            else:
                cash_flow = -size * price - fees
            out[i, col] = add_nb(out[i, col], cash_flow)
            position_now = new_position_now
    return out
```

## sum_grouped_nb
指定分组 `group_lens` 对 `a` 进行归约：组中求和。

```python
@njit(cache=True)
def sum_grouped_nb(a: tp.Array2d, group_lens: tp.Array1d) -> tp.Array2d:
    check_group_lens_nb(group_lens, a.shape[1])

    out = np.empty((a.shape[0], len(group_lens)), dtype=np.float64)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        out[:, group] = np.sum(a[:, from_col:to_col], axis=1)
        from_col = to_col
    return out
```

## cash_flow_grouped_nb
```python
@njit(cache=True)
def cash_flow_grouped_nb(cash_flow: tp.Array2d, group_lens: tp.Array1d) -> tp.Array2d:
    return sum_grouped_nb(cash_flow, group_lens)
```

## init_cash_grouped_nb
指定分组 `group_lens` 对 `init_cash` 进行归约：组中求和。

```python
@njit(cache=True)
def init_cash_grouped_nb(init_cash: tp.Array1d, group_lens: tp.Array1d, cash_sharing: bool) -> tp.Array1d:
    if cash_sharing:
        return init_cash
    out = np.empty(group_lens.shape, dtype=np.float64)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        cash_sum = 0.
        for col in range(from_col, to_col):
            cash_sum += init_cash[col]
        out[group] = cash_sum
        from_col = to_col
    return out
```

## init_cash_nb
基于 `init_cash` 和 `group_lens` 生成资金序列。
- 非资金共享 `cash_sharing == False` 时，直接返回 `init_cash`
- 否则：`init_cash` 与 `group_lens` 中的对应项分别是返回资金序列中的金额和项数

```python
@njit(cache=True)
def init_cash_nb(init_cash: tp.Array1d, group_lens: tp.Array1d, cash_sharing: bool) -> tp.Array1d:
    if not cash_sharing:
        return init_cash
    group_lens_cs = np.cumsum(group_lens)
    out = np.full(group_lens_cs[-1], np.nan, dtype=np.float64)
    out[group_lens_cs - group_lens] = init_cash
    out = generic_nb.ffill_1d_nb(out)
    return out
```

## cash_nb
基于资产初始资金 `init_cash`，根据现金 `cash_flow` 计算每个资产逐订单的累积现金。

```python
@njit(cache=True)
def cash_nb(cash_flow: tp.Array2d, init_cash: tp.Array1d) -> tp.Array2d:
    out = np.empty_like(cash_flow)
    for col in range(cash_flow.shape[1]):
        for i in range(cash_flow.shape[0]):
            cash_now = init_cash[col] if i == 0 else out[i - 1, col]
            out[i, col] = add_nb(cash_now, cash_flow[i, col])
    return out
```

## cash_in_sim_order_nb
基于资产初始资金 `init_cash_grouped`，根据现金 `cash_flow` 以及调用序列 `call_seq` 计算每个资产逐订单的累积现金。

```python
@njit(cache=True)
def cash_in_sim_order_nb(cash_flow: tp.Array2d,
                         group_lens: tp.Array1d,
                         init_cash_grouped: tp.Array1d,
                         call_seq: tp.Array2d) -> tp.Array2d:
    check_group_lens_nb(group_lens, cash_flow.shape[1])

    out = np.empty_like(cash_flow)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        group_len = to_col - from_col
        cash_now = init_cash_grouped[group]
        for i in range(cash_flow.shape[0]):
            for k in range(group_len):
                col = from_col + call_seq[i, from_col + k]
                cash_now = add_nb(cash_now, cash_flow[i, col])
                out[i, col] = cash_now
        from_col = to_col
    return out
```

## cash_grouped_nb
基于资产初始资金 `init_cash_grouped`，根据现金 `cash_flow_grouped` 计算每个资产的累积现金。

```python
@njit(cache=True)
def cash_grouped_nb(target_shape: tp.Shape,
                    cash_flow_grouped: tp.Array2d,
                    group_lens: tp.Array1d,
                    init_cash_grouped: tp.Array1d) -> tp.Array2d:
    check_group_lens_nb(group_lens, target_shape[1])

    out = np.empty_like(cash_flow_grouped)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        cash_now = init_cash_grouped[group]
        for i in range(cash_flow_grouped.shape[0]):
            flow_value = cash_flow_grouped[i, group]
            cash_now = add_nb(cash_now, flow_value)
            out[i, group] = cash_now
        from_col = to_col
    return out
```

# 绩效分析

## asset_value_nb
返回 $close\left[ {t,{\rm{ }}i} \right]{\rm{ }} \times {\rm{ }}assets\left[ {t,{\rm{ }}i} \right]$
```python
@njit(cache=True)
def asset_value_nb(close: tp.Array2d, assets: tp.Array2d) -> tp.Array2d:
    return close * assets
```

## asset_value_grouped_nb

### 源码
指定分组 `group_lens` 对 `asset_value` 进行归约：组中求和。
```python
@njit(cache=True)
def asset_value_grouped_nb(asset_value: tp.Array2d, group_lens: tp.Array1d) -> tp.Array2d:
    return sum_grouped_nb(asset_value, group_lens)
```

## value_in_sim_order_nb
按照时间步，根据订单执行顺序（`call_seq`）计算每个资产组的总价值。

### 逻辑
- 遍历每个资产组
  - 初始化
    - 当前组的累计资产价值 `asset_value_now = 0`
    - 当前组的 NaN 计数器 `since_last_nan = group_len`
  - 遍历所有（时间步，当前资产组的所有列）
    - 计算当前时间步 `i` 以及列 `col`（根据执行顺序 `call_seq`）
    - 如果不属于第一个时间步 `j >= group_len`
      - 从 `asset_value_now` 中减去上一时间步的 `asset_value` 中的对应项（如果不是 `NaN`）
    - 如果当前资产价值（`asset_value` 中的对应项）是 `NaN`
      - 重置 NaN 计数器 `since_last_nan = 0`
    - 否则累加到 `asset_value_now`
    - 如果 `since_last_nan < group_len`
      - ` out[i, col] = NaN`
    - 否则
      - `out[i, col] = cash[i, col] + asset_value_now`

### 源码
```python
@njit(cache=True)
def value_in_sim_order_nb(cash: tp.Array2d,
                          asset_value: tp.Array2d,
                          group_lens: tp.Array1d,
                          call_seq: tp.Array2d) -> tp.Array2d:
    check_group_lens_nb(group_lens, cash.shape[1])

    out = np.empty_like(cash)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        group_len = to_col - from_col
        asset_value_now = 0.
        since_last_nan = group_len
        for j in range(cash.shape[0] * group_len):
            i = j // group_len
            col = from_col + call_seq[i, from_col + j % group_len]
            if j >= group_len:
                last_j = j - group_len
                last_i = last_j // group_len
                last_col = from_col + call_seq[last_i, from_col + last_j % group_len]
                if not np.isnan(asset_value[last_i, last_col]):
                    asset_value_now -= asset_value[last_i, last_col]
            if np.isnan(asset_value[i, col]):
                since_last_nan = 0
            else:
                asset_value_now += asset_value[i, col]
            if since_last_nan < group_len:
                out[i, col] = np.nan
            else:
                out[i, col] = cash[i, col] + asset_value_now
            since_last_nan += 1

        from_col = to_col
    return out
```

## value_nb
```python
@njit(cache=True)
def value_nb(cash: tp.Array2d, asset_value: tp.Array2d) -> tp.Array2d:
    return cash + asset_value
```

## total_profit_nb
计算每个资产组的盈利。

### 逻辑
- 遍历每个资产组
  - 遍历该组的所有订单
    - 从 `order_records` 中提取对应的订单信息
    - 更新资产数 `assets[col]`
      - 如果是买入 `OrderSide.Buy`：`assets[col]` 加上订单数
      - 否则（`OrderSide.Sell`） ：`assets[col]` 减去订单数
    - 更新现金 `cash[col]`
      - 如果是买入 `OrderSide.Buy`：`assets[col]` 减去 *订单大小 × 订单价格 + 手续费*
      - 否则（`OrderSide.Sell`）：`assets[col]` 加上 *订单大小 × 订单价格 + 手续费*
- 计算最终盈利 `total_profit = cash + assets * close[-1, :]` 并返回

### 源码
```python
@njit(cache=True)
def total_profit_nb(target_shape: tp.Shape,
                    close: tp.Array2d,
                    order_records: tp.RecordArray,
                    col_map: tp.ColMap) -> tp.Array1d:
    col_idxs, col_lens = col_map
    col_start_idxs = np.cumsum(col_lens) - col_lens
    assets = np.full(target_shape[1], 0., dtype=np.float64)
    cash = np.full(target_shape[1], 0., dtype=np.float64)
    zero_mask = np.full(target_shape[1], False, dtype=np.bool_)

    for col in range(col_lens.shape[0]):
        col_len = col_lens[col]
        if col_len == 0:
            zero_mask[col] = True
            continue
        last_id = -1

        for c in range(col_len):
            oidx = col_idxs[col_start_idxs[col] + c]
            record = order_records[oidx]

            if record['id'] < last_id:
                raise ValueError("id must come in ascending order per column")
            last_id = record['id']

            # Fill assets
            if record['side'] == OrderSide.Buy:
                order_size = record['size']
                assets[col] = add_nb(assets[col], order_size)
            else:
                order_size = record['size']
                assets[col] = add_nb(assets[col], -order_size)

            # Fill cash balance
            if record['side'] == OrderSide.Buy:
                order_cash = record['size'] * record['price'] + record['fees']
                cash[col] = add_nb(cash[col], -order_cash)
            else:
                order_cash = record['size'] * record['price'] - record['fees']
                cash[col] = add_nb(cash[col], order_cash)

    total_profit = cash + assets * close[-1, :]
    total_profit[zero_mask] = 0.
    return total_profit
```

## total_profit_grouped_nb
根据分组信息 `group_lens` 计算 `total_profit` 中各组的和。
```python
@njit(cache=True)
def total_profit_grouped_nb(total_profit: tp.Array1d, group_lens: tp.Array1d) -> tp.Array1d:
    check_group_lens_nb(group_lens, total_profit.shape[0])

    out = np.empty(len(group_lens), dtype=np.float64)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        out[group] = np.sum(total_profit[from_col:to_col])
        from_col = to_col
    return out
```

## final_value_nb
```python
@njit(cache=True)
def final_value_nb(total_profit: tp.Array1d, init_cash: tp.Array1d) -> tp.Array1d:
    return total_profit + init_cash
```

## total_return_nb
```python
@njit(cache=True)
def total_return_nb(total_profit: tp.Array1d, init_cash: tp.Array1d) -> tp.Array1d:
    return total_profit / init_cash
```

## returns_in_sim_order_nb
计算每个资产组的收益率变化。

逻辑
- 遍历每个资产组
  - 获取该组初始现金 `input_value`
  - 遍历（时间步，该组内订单）
    - 从 `value_iso` 获取该位置的输出价值 `output_value`
    - 计算收益率 `(output_value - input_value) / output_value` 并记录到 `out` 中对应位置
    - `input_value = output_value`
- 返回 `out`

### 源码
```python
@njit(cache=True)
def returns_in_sim_order_nb(value_iso: tp.Array2d,
                            group_lens: tp.Array1d,
                            init_cash_grouped: tp.Array1d,
                            call_seq: tp.Array2d) -> tp.Array2d:
    check_group_lens_nb(group_lens, value_iso.shape[1])

    out = np.empty_like(value_iso)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        group_len = to_col - from_col
        input_value = init_cash_grouped[group]
        for j in range(value_iso.shape[0] * group_len):
            i = j // group_len
            col = from_col + call_seq[i, from_col + j % group_len]
            output_value = value_iso[i, col]
            out[i, col] = returns_nb.get_return_nb(input_value, output_value)
            input_value = output_value
        from_col = to_col
    return out
```

## asset_returns_nb
计算资产的收益率变化。

逻辑
- 遍历每列
  - 遍历每行
    - 遍历（时间步，该组内订单）
      - 从 `asset_value` 获取输入价值 `input_value`
      - 从 `asset_value` 和 `cash_flow` 获取并计算输出价值 `output_value`
      - 计算收益率 `(output_value - input_value) / output_value` 并记录到 `out` 中对应位置
- 返回 `out`

### 源码
```python
@njit(cache=True)
def asset_returns_nb(cash_flow: tp.Array2d, asset_value: tp.Array2d) -> tp.Array2d:
    out = np.empty_like(cash_flow)
    for col in range(cash_flow.shape[1]):
        for i in range(cash_flow.shape[0]):
            input_value = 0. if i == 0 else asset_value[i - 1, col]
            output_value = asset_value[i, col] + cash_flow[i, col]
            out[i, col] = returns_nb.get_return_nb(input_value, output_value)
    return out
```

## benchmark_value_nb
```python
@njit(cache=True)
def benchmark_value_nb(close: tp.Array2d, init_cash: tp.Array1d) -> tp.Array2d:
    return close / close[0] * init_cash
```

## benchmark_value_grouped_nb
计算每个资产组的 $$\frac{总投资}{资产数} \cdot \left( {\frac{第一个资产的当前收盘价}{第一个资产的初始收盘价} +  \cdots  + \frac{最后一个资产的当前收盘价}{最后一个资产的初始收盘价}} \right)$$


### 源码
```python
@njit(cache=True)
def benchmark_value_grouped_nb(close: tp.Array2d, group_lens: tp.Array1d, init_cash_grouped: tp.Array1d) -> tp.Array2d:
    check_group_lens_nb(group_lens, close.shape[1])

    out = np.empty((close.shape[0], len(group_lens)), dtype=np.float64)
    from_col = 0
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        group_len = to_col - from_col
        col_init_cash = init_cash_grouped[group] / group_len
        close_norm = close[:, from_col:to_col] / close[0, from_col:to_col]
        out[:, group] = col_init_cash * np.sum(close_norm, axis=1)
        from_col = to_col
    return out
```

## total_benchmark_return_nb
计算 `benchmark_value` 的收益率。
```python
@njit(cache=True)
def total_benchmark_return_nb(benchmark_value: tp.Array2d) -> tp.Array1d:
    out = np.empty(benchmark_value.shape[1], dtype=np.float64)
    for col in range(benchmark_value.shape[1]):
        out[col] = returns_nb.get_return_nb(benchmark_value[0, col], benchmark_value[-1, col])
    return out
```

## gross_exposure_nb
计算 `out`，其中
$$out\left[ {i,col} \right] = \frac{{asset\_value\left[ {i,col} \right]}}{{asset\_value\left[ {i,col} \right] + cash\left[ {i,col} \right]}}$$
```python
@njit(cache=True)
def gross_exposure_nb(asset_value: tp.Array2d, cash: tp.Array2d) -> tp.Array2d:
    out = np.empty(asset_value.shape, dtype=np.float64)
    for col in range(out.shape[1]):
        for i in range(out.shape[0]):
            denom = add_nb(asset_value[i, col], cash[i, col])
            if denom == 0:
                out[i, col] = 0.
            else:
                out[i, col] = asset_value[i, col] / denom
    return out
```